<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/NBA_Prop_Hit_Sheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NBA 3:1 props.csv to NBA 3:1 props.csv


In [ ]:
import pandas as pd

# Get uploaded filename dynamically
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Raw Columns:")
print(df.columns.tolist())

df.head()

Raw Columns:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaN,PLAYER,GAME,PROP,OUTCOME,ODDS,AVG,IM PROB %,L5,L10,2025,HM/AW,H2H,DVP L5,DVP L10,DVP 2025,DVP HM/AW
1,NaN,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,BET\n-110,-1102,52.4%,100%,100%,100%,100%,100%,100%,100%,100%,100%
2,NaN,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,BET\n+105,1052,48.8%,100%,100%,100%,100%,100%,100%,100%,100%,100%
3,NaN,Jarrett Allen,CLE at BKN,Rebounds,o0.5,BET\n-145,-1452,59.2%,100%,100%,100%,100%,100%,100%,100%,98.3%,96.6%
4,NaN,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,BET\n-129,-1304,56.3%,100%,90%,97.7%,100%,100%,20%,30%,15.5%,13.3%


In [ ]:
import numpy as np
import re

# Promote first row to header
df.columns = df.iloc[0]
df = df.drop(index=0).reset_index(drop=True)

df.columns = df.columns.astype(str).str.strip()

# Drop empty columns
df = df.loc[:, df.columns.notna()]
df = df.dropna(axis=1, how='all')

# Extract American odds from "BET\n-169"
def extract_american_odds(x):
    if pd.isna(x):
        return np.nan
    match = re.search(r'([+-]\d+)', str(x))
    return float(match.group(1)) if match else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american_odds)

# Normalize percentage columns
pct_cols = ["IM PROB %","L5","L10","2025","HM/AW","H2H",
            "DVP L5","DVP L10","DVP 2025","DVP HM/AW"]

for c in pct_cols:
    if c in df.columns:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace("%","", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

print("Cleaned Columns:")
print(df.columns.tolist())

df.head()

In [ ]:
 import numpy as np
import pandas as pd

# ---------- Helpers ----------
def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

def american_to_decimal(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return 1.0 + (100.0/(-o))
    return 1.0 + (o/100.0)

def kelly_fraction(p, dec_odds):
    # For decimal odds d: b = d - 1
    if pd.isna(p) or pd.isna(dec_odds):
        return np.nan
    b = dec_odds - 1.0
    q = 1.0 - p
    if b <= 0:
        return np.nan
    f = (b*p - q) / b
    return max(0.0, f)

def safe_mean(row, cols):
    vals = [row[c] for c in cols if c in row.index and pd.notna(row[c])]
    return np.mean(vals) if len(vals) else np.nan

# ---------- 1) Book probabilities ----------
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# ---------- 2) Weighted True Probability (NBA tuned) ----------
# NBA: more stable; avoid overfitting L5. Use L10/Season heavier.
W = {
    "L5": 0.20,
    "L10": 0.30,
    "2025": 0.25,
    "HM/AW": 0.10,
    "H2H": 0.05,
    "IM PROB %": 0.10,   # optional “market-like” signal; keeps us anchored
}

def weighted_prob(row):
    num, den = 0.0, 0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w * float(v)
            den += w
    return num/den if den > 0 else np.nan

df["P_TREND"] = df.apply(weighted_prob, axis=1)

# ---------- 3) DVP Matchup Layer (capped) ----------
# Use available dvp cols; compute mean and cap its influence so it can't dominate.
dvp_cols = [c for c in ["DVP L5","DVP L10","DVP 2025","DVP HM/AW"] if c in df.columns]
df["P_DVP_RAW"] = df.apply(lambda r: safe_mean(r, dvp_cols), axis=1)

def dvp_adjust(p_dvp):
    if pd.isna(p_dvp):
        return 0.0
    # centered around 0.50; cap impact to +/- 0.06 for NBA
    adj = float(p_dvp) - 0.50
    return float(np.clip(adj, -0.06, 0.06))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust)

# ---------- 4) Final Model Probability ----------
df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)

# ---------- 5) Edge + EV ----------
df["EDGE"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"] * (df["DEC_ODDS"] - 1.0) - (1.0 - df["MODEL_PROB"])

# ---------- 6) Kelly sizing ----------
df["KELLY_FULL"] = df.apply(lambda r: kelly_fraction(r["MODEL_PROB"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF"] = 0.5 * df["KELLY_FULL"]

# Convert to "units" with a simple bankroll convention:
# If 1u = 1% bankroll, units = kelly_half * 100
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"] * 100).round(2)
df["KELLY_FULL_UNITS"] = (df["KELLY_FULL"] * 100).round(2)

# ---------- 7) Outputs ----------
# Top 10 Most Likely (by MODEL_PROB)
top10_likely = df.sort_values("MODEL_PROB", ascending=False).head(10)

# Top 10 Highest EV (by EV_1U)
top10_ev = df.sort_values("EV_1U", ascending=False).head(10)

print("TOP 10 MOST LIKELY")
display(top10_likely[[
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"
]])

print("\nTOP 10 HIGHEST EV")
display(top10_ev[[
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"
]])

# Save files
df.to_csv("nba_prop_engine_v1_full_output.csv", index=False)
top10_likely.to_csv("nba_prop_engine_v1_top10_most_likely.csv", index=False)
top10_ev.to_csv("nba_prop_engine_v1_top10_highest_ev.csv", index=False)

print("\nSaved: nba_prop_engine_v1_full_output.csv")
print("Saved: nba_prop_engine_v1_top10_most_likely.csv")
print("Saved: nba_prop_engine_v1_top10_highest_ev.csv")

In [ ]:
# =============================
# NBA PROP ENGINE — MASTER LOAD
# =============================

from google.colab import files
import pandas as pd
import numpy as np
import re

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

# Promote first row to header
df.columns = df.iloc[0]
df = df.drop(index=0).reset_index(drop=True)
df.columns = df.columns.astype(str).str.strip()

# Extract American odds
def extract_american_odds(x):
    if pd.isna(x):
        return np.nan
    match = re.search(r'([+-]\d+)', str(x))
    return float(match.group(1)) if match else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american_odds)

# Normalize % fields
pct_cols = ["IM PROB %","L5","L10","2025","HM/AW","H2H",
            "DVP L5","DVP L10","DVP 2025","DVP HM/AW"]

for c in pct_cols:
    if c in df.columns:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace("%","", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

print("NBA data loaded and cleaned.")
print("Rows:", len(df))
df.head()

In [ ]:
# Drop junk columns created by blank header cells
df = df.loc[:, ~df.columns.isna()]  # drop NaN column names
df = df.loc[:, ~df.columns.astype(str).str.lower().isin(["nan"])]  # drop literal "nan"
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]  # drop Unnamed: 0, etc.

# Strip column names again (safe)
df.columns = df.columns.astype(str).str.strip()

print("Columns now:", df.columns.tolist())
df.head()

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
from datetime import datetime, timezone

ODDS_API_KEY = "PASTE_YOUR_KEY_HERE"

def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

def fetch_odds_api_nba_markets(regions="us", markets="player_points,player_rebounds,player_assists,player_threes,player_points_rebounds_assists"):
    url = "https://api.the-odds-api.com/v4/sports/basketball_nba/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": "american",
        "dateFormat": "iso"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

odds_json = fetch_odds_api_nba_markets()
print("Games returned:", len(odds_json))

In [ ]:
import numpy as np
import pandas as pd

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return 1.0 + (100.0/(-o)) if o < 0 else 1.0 + (o/100.0)

def kelly_fraction(p, dec_odds):
    if pd.isna(p) or pd.isna(dec_odds): return np.nan
    b = dec_odds - 1.0
    q = 1.0 - p
    if b <= 0: return np.nan
    f = (b*p - q)/b
    return max(0.0, f)

# Book probs from your CSV odds
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# NBA weights (avoid overfitting L5)
W = {"L5":0.20, "L10":0.30, "2025":0.25, "HM/AW":0.10, "H2H":0.05, "IM PROB %":0.10}

def weighted_prob(row):
    num=den=0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w*float(v); den += w
    return num/den if den>0 else np.nan

df["P_TREND"] = df.apply(weighted_prob, axis=1)

# DVP capped (NBA)
dvp_cols = [c for c in ["DVP L5","DVP L10","DVP 2025","DVP HM/AW"] if c in df.columns]
df["P_DVP_RAW"] = df[dvp_cols].mean(axis=1, skipna=True)

def dvp_adjust(p):
    if pd.isna(p): return 0.0
    return float(np.clip(p - 0.50, -0.06, 0.06))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust)

df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)

df["EDGE"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1.0) - (1.0-df["MODEL_PROB"])

df["KELLY_FULL"] = df.apply(lambda r: kelly_fraction(r["MODEL_PROB"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF"] = 0.5*df["KELLY_FULL"]
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"]*100).round(2)

top10_likely = df.sort_values("MODEL_PROB", ascending=False).head(10)
top10_ev     = df.sort_values("EV_1U", ascending=False).head(10)

print("TOP 10 MOST LIKELY")
display(top10_likely[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB","BOOK_PROB","EDGE","KELLY_HALF_UNITS"]])

print("\nTOP 10 HIGHEST EV")
display(top10_ev[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"]])

df.to_csv("nba_prop_engine_v1_full_output.csv", index=False)
top10_likely.to_csv("nba_prop_engine_v1_top10_most_likely.csv", index=False)
top10_ev.to_csv("nba_prop_engine_v1_top10_highest_ev.csv", index=False)

print("\nSaved: nba_prop_engine_v1_full_output.csv")
print("Saved: nba_prop_engine_v1_top10_most_likely.csv")
print("Saved: nba_prop_engine_v1_top10_highest_ev.csv")

In [ ]:
import requests, re, numpy as np, pandas as pd
from functools import lru_cache

ESPN_NBA_SUMMARY  = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary"
ESPN_NBA_TEAMS    = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
ESPN_NBA_SCHEDULE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/schedule"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=4096)
def get_team_list():
    return get_json(ESPN_NBA_TEAMS)

@lru_cache(maxsize=4096)
def get_schedule(team_id: str):
    return get_json(ESPN_NBA_SCHEDULE.format(team_id=str(team_id)))

@lru_cache(maxsize=8192)
def get_summary(event_id: str):
    return get_json(ESPN_NBA_SUMMARY, params={"event": str(event_id)})

def norm_name(s):
    s = str(s).upper().strip()
    s = re.sub(r"\b(JR|SR|II|III|IV)\b", "", s)
    s = re.sub(r"[^A-Z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_minutes(mmss):
    if mmss is None or (isinstance(mmss,float) and np.isnan(mmss)):
        return np.nan
    s = str(mmss).strip()
    if ":" in s:
        m, sec = s.split(":", 1)
        try: return float(m) + float(sec)/60.0
        except: return np.nan
    try: return float(s)
    except: return np.nan

In [ ]:
# Build abbrev -> team_id map from ESPN
teams_json = get_team_list()

abbrev_to_id = {}
for sp in teams_json.get("sports", []):
    for lg in sp.get("leagues", []):
        for tm in lg.get("teams", []):
            t = tm.get("team", {})
            ab = (t.get("abbreviation") or "").upper()
            tid = t.get("id")
            if ab and tid:
                abbrev_to_id[ab] = str(tid)

print("ESPN teams mapped:", len(abbrev_to_id))
print("Example:", list(abbrev_to_id.items())[:10])

# Parse GAME into away/home abbrevs
def parse_game_abbrevs(game_str):
    s = str(game_str).upper().strip()
    if " AT " in s:
        away, home = s.split(" AT ", 1)
    elif " VS " in s:
        away, home = s.split(" VS ", 1)
    else:
        return None, None
    return away.strip(), home.strip()

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].apply(parse_game_abbrevs))

df["AWAY_TEAM_ID"] = df["AWAY_ABBR"].map(abbrev_to_id)
df["HOME_TEAM_ID"] = df["HOME_ABBR"].map(abbrev_to_id)

missing_team = df[df["AWAY_TEAM_ID"].isna() | df["HOME_TEAM_ID"].isna()][["GAME","AWAY_ABBR","HOME_ABBR","AWAY_TEAM_ID","HOME_TEAM_ID"]].drop_duplicates()
print("Games with missing ESPN team ids:", len(missing_team))
display(missing_team.head(20))

In [ ]:
def completed_event_ids_from_schedule(team_id, max_games=15):
    j = get_schedule(team_id)
    events = j.get("events", []) or []
    evs = []

    for e in events:
        # event id
        eid = e.get("id") or None
        if eid is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            eid = e["competitions"][0].get("id")

        # status
        status = None
        if isinstance(e.get("status"), dict):
            status = (e["status"].get("type") or {}).get("name") or (e["status"].get("type") or {}).get("state")
        if status is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            st = (e["competitions"][0].get("status") or {}).get("type") or {}
            status = st.get("name") or st.get("state")

        st = str(status).upper() if status else ""
        if ("FINAL" not in st) and ("POST" not in st):
            continue

        dt = e.get("date")
        if dt is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            dt = e["competitions"][0].get("date")

        evs.append({"eventId": str(eid), "date": dt})

    # sort most recent first
    evs = sorted(evs, key=lambda x: pd.to_datetime(x["date"], utc=True, errors="coerce"), reverse=True)
    evs = [x for x in evs if x["eventId"] not in ["None","nan",""]]
    return [x["eventId"] for x in evs[:max_games]]

team_ids = pd.unique(pd.concat([df["AWAY_TEAM_ID"], df["HOME_TEAM_ID"]]).dropna()).tolist()
team_recent_events = {}

for tid in team_ids:
    try:
        team_recent_events[str(tid)] = completed_event_ids_from_schedule(str(tid), max_games=15)
    except Exception as e:
        team_recent_events[str(tid)] = []
        print("Schedule fail team", tid, e)

print("Teams pulled:", len(team_recent_events))
print("Example counts:", {k: len(v) for k,v in list(team_recent_events.items())[:5]})

In [ ]:
def extract_minutes_for_player_in_event(event_id: str, player_norm: str):
    j = get_summary(event_id)
    box = j.get("boxscore", {})
    players = box.get("players", [])
    if not players:
        return np.nan

    for team_block in players:
        for stat_group in team_block.get("statistics", []):
            labels = stat_group.get("labels", [])
            if "MIN" not in labels:
                continue
            min_idx = labels.index("MIN")

            for a in stat_group.get("athletes", []):
                ath = a.get("athlete", {}) or {}
                nm = norm_name(ath.get("displayName",""))
                if nm != player_norm:
                    continue
                stats = a.get("stats", [])
                mm = stats[min_idx] if min_idx < len(stats) else None
                return parse_minutes(mm)

    return np.nan

# Build unique player keys from your sheet
df["PLAYER_NORM"] = df["PLAYER"].apply(norm_name)
unique_players = df[["PLAYER","PLAYER_NORM","AWAY_TEAM_ID","HOME_TEAM_ID"]].drop_duplicates()

N_BACK = 10
hist_rows = []

for _, r in unique_players.iterrows():
    pnorm = r["PLAYER_NORM"]
    away_tid = str(r["AWAY_TEAM_ID"]) if pd.notna(r["AWAY_TEAM_ID"]) else None
    home_tid = str(r["HOME_TEAM_ID"]) if pd.notna(r["HOME_TEAM_ID"]) else None

    candidate_events = []
    for tid in [away_tid, home_tid]:
        if tid and tid in team_recent_events:
            candidate_events += team_recent_events[tid][:N_BACK]

    # dedupe preserve order
    seen=set()
    candidate_events = [x for x in candidate_events if not (x in seen or seen.add(x))]
    candidate_events = candidate_events[:N_BACK]

    mins_list = []
    for eid in candidate_events:
        try:
            m = extract_minutes_for_player_in_event(eid, pnorm)
            if pd.notna(m):
                mins_list.append(m)
        except:
            continue

    if len(mins_list) == 0:
        hist_rows.append({"PLAYER_NORM": pnorm, "MIN_L5": np.nan, "MIN_L10": np.nan, "MIN_SD_L10": np.nan, "GAMES_FOUND": 0})
    else:
        arr = np.array(mins_list, dtype=float)
        hist_rows.append({
            "PLAYER_NORM": pnorm,
            "MIN_L5": float(np.nanmean(arr[:5])),
            "MIN_L10": float(np.nanmean(arr[:10])),
            "MIN_SD_L10": float(np.nanstd(arr[:10])) if len(arr[:10]) >= 2 else np.nan,
            "GAMES_FOUND": int(len(arr))
        })

mins_hist_df = pd.DataFrame(hist_rows)
print("Minutes rows:", len(mins_hist_df))
display(mins_hist_df.head(15))

# merge into df
df = df.merge(mins_hist_df, on="PLAYER_NORM", how="left")
print("Rows with GAMES_FOUND>0:", int((df["GAMES_FOUND"].fillna(0)>0).sum()), "of", len(df))

In [ ]:
def role_adj(min_l10, sd_l10):
    if pd.isna(min_l10):
        base = 0.0
    elif min_l10 >= 34:
        base = +0.030
    elif min_l10 >= 30:
        base = +0.015
    elif min_l10 >= 24:
        base = +0.005
    elif min_l10 >= 18:
        base = -0.020
    else:
        base = -0.045

    if pd.isna(sd_l10):
        stab = 0.0
    elif sd_l10 >= 8:
        stab = -0.020
    elif sd_l10 >= 5:
        stab = -0.010
    else:
        stab = 0.0

    return base + stab

df["ADJ_ROLE"] = df.apply(lambda r: role_adj(r["MIN_L10"], r["MIN_SD_L10"]), axis=1)

df["DNP_RATE_L10"] = (10 - df["GAMES_FOUND"].fillna(0)).clip(0,10) / 10.0
df["MIN_TREND_L5_L10"] = df["MIN_L5"] - df["MIN_L10"]

def inj_proxy(dnp, trend):
    pen = 0.0
    if pd.notna(dnp) and dnp >= 0.3: pen -= 0.02
    if pd.notna(dnp) and dnp >= 0.5: pen -= 0.04
    if pd.notna(trend) and trend <= -4: pen -= 0.02
    if pd.notna(trend) and trend <= -7: pen -= 0.04
    return pen

df["ADJ_INJ_PROXY"] = df.apply(lambda r: inj_proxy(r["DNP_RATE_L10"], r["MIN_TREND_L5_L10"]), axis=1)

# Final ESPN-adjusted model probability
df["MODEL_PROB_ESPN_V1"] = (df["MODEL_PROB"] + df["ADJ_ROLE"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)

# Top 10 Most Likely (role-verified; dedupe by player)
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V1", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10 = df_rank.head(10).copy()
top10.insert(0, "RANK", range(1, len(top10)+1))

print(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB","ADJ_ROLE","ADJ_INJ_PROXY","MODEL_PROB_ESPN_V1",
    "MIN_L10","MIN_SD_L10","GAMES_FOUND","MIN_TREND_L5_L10","DNP_RATE_L10"
]].to_string(index=False))

top10.to_csv("nba_prop_engine_v1_top10_most_likely_ESPN_ROLE.csv", index=False)
df.to_csv("nba_prop_engine_v1_full_ESPN_role.csv", index=False)
print("\nSaved: nba_prop_engine_v1_top10_most_likely_ESPN_ROLE.csv")
print("Saved: nba_prop_engine_v1_full_ESPN_role.csv")

In [ ]:
def discord_top10(top10_df):
    lines = ["NBA PROP ENGINE v1 — TOP 10 MOST LIKELY (ESPN ROLE)", "—"]
    for r in top10_df.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME}\n"
            f"Model: {r.MODEL_PROB_ESPN_V1*100:.1f}% | MIN_L10: {r.MIN_L10:.1f} | Trend: {r.MIN_TREND_L5_L10:+.1f}"
        )
    return "\n".join(lines)

print(discord_top10(top10))

In [ ]:
# --- Reduce DVP influence (NBA more efficient than college) ---
def dvp_adjust_nba(p):
    if pd.isna(p): return 0.0
    return float(np.clip(p - 0.50, -0.04, 0.04))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust_nba)

df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.95)


# --- Stronger Role Penalty ---
def role_adj_v2(min_l10, sd_l10):
    if pd.isna(min_l10):
        base = -0.02
    elif min_l10 >= 35:
        base = +0.035
    elif min_l10 >= 32:
        base = +0.020
    elif min_l10 >= 28:
        base = +0.010
    elif min_l10 >= 24:
        base = -0.010
    elif min_l10 >= 20:
        base = -0.035
    else:
        base = -0.060

    # volatility penalty
    if pd.isna(sd_l10):
        stab = 0.0
    elif sd_l10 >= 8:
        stab = -0.025
    elif sd_l10 >= 6:
        stab = -0.015
    else:
        stab = 0.0

    return base + stab

df["ADJ_ROLE"] = df.apply(lambda r: role_adj_v2(r["MIN_L10"], r["MIN_SD_L10"]), axis=1)

df["MODEL_PROB_ESPN_V2"] = (
    df["MODEL_PROB"] + df["ADJ_ROLE"] + df["ADJ_INJ_PROXY"]
).clip(0.02, 0.90)   # hard NBA realism cap


# Re-rank
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V2", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10_v2 = df_rank.head(10).copy()
top10_v2.insert(0, "RANK", range(1, len(top10_v2)+1))

print(top10_v2[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB_ESPN_V2",
    "MIN_L10","MIN_SD_L10","MIN_TREND_L5_L10"
]].to_string(index=False))

In [ ]:
# --- Hard minutes ceiling governor ---
def minutes_ceiling(prob, min_l10):
    if pd.isna(min_l10):
        return min(prob, 0.75)
    if min_l10 < 22:
        return min(prob, 0.70)
    if min_l10 < 24:
        return min(prob, 0.75)
    return prob

df["MODEL_PROB_ESPN_V3"] = df.apply(
    lambda r: minutes_ceiling(r["MODEL_PROB_ESPN_V2"], r["MIN_L10"]),
    axis=1
)

# Re-rank
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V3", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10_v3 = df_rank.head(10).copy()
top10_v3.insert(0, "RANK", range(1, len(top10_v3)+1))

print(top10_v3[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB_ESPN_V3",
    "MIN_L10","MIN_SD_L10"
]].to_string(index=False))

In [ ]:
# Recalculate book probabilities
def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return 1.0 + (100.0/(-o)) if o < 0 else 1.0 + (o/100.0)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

df["EDGE_VS_BOOK"] = df["MODEL_PROB_ESPN_V3"] - df["BOOK_PROB"]

df["EV_1U"] = df["MODEL_PROB_ESPN_V3"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB_ESPN_V3"])

def kelly(p, d):
    if pd.isna(p) or pd.isna(d): return np.nan
    b = d-1
    q = 1-p
    if b <= 0: return np.nan
    return max(0, (b*p - q)/b)

df["KELLY_HALF"] = 0.5 * df.apply(lambda r: kelly(r["MODEL_PROB_ESPN_V3"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"]*100).round(2)

# Final ranking
final_rank = df[df["GAMES_FOUND"].fillna(0)>=3].copy()
final_rank = final_rank.sort_values("MODEL_PROB_ESPN_V3", ascending=False)
final_rank = final_rank.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10 = final_rank.head(10).copy()
final_top10.insert(0,"RANK",range(1,len(final_top10)+1))

print(final_top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS",
    "MODEL_PROB_ESPN_V3",
    "BOOK_PROB",
    "EDGE_VS_BOOK",
    "KELLY_HALF_UNITS"
]].to_string(index=False))

In [ ]:
def discord_ready(df_block):
    lines = ["NBA PROP ENGINE v1.3 — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.MODEL_PROB_ESPN_V3*100:.1f}% | Edge: {r.EDGE_VS_BOOK*100:+.1f} pts | Half-Kelly: {r.KELLY_HALF_UNITS}u"
        )
    return "\n".join(lines)

print(discord_ready(final_top10))

In [ ]:
def discord_hit_sheet(df_block):
    lines = ["NBA PROP ENGINE — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.MODEL_PROB_CAL*100:.1f}% | Edge: {r.EDGE_CAL*100:+.1f} pts | MIN_L10: {r.MIN_L10:.1f}"
        )
    return "\n".join(lines)

print(discord_hit_sheet(final_top10))

In [ ]:
# Ensure book prob exists
def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)

# Final probability (no compression)
df["FINAL_PROB"] = df["MODEL_PROB_ESPN_V3"]

# Edge
df["FINAL_EDGE"] = df["FINAL_PROB"] - df["BOOK_PROB"]

# Re-rank
final_rank = df[df["GAMES_FOUND"].fillna(0)>=3].copy()
final_rank = final_rank.sort_values("FINAL_PROB", ascending=False)
final_rank = final_rank.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10 = final_rank.head(10).copy()
final_top10.insert(0, "RANK", range(1, len(final_top10)+1))

print(final_top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS",
    "FINAL_PROB",
    "FINAL_EDGE",
    "MIN_L10"
]].to_string(index=False))

In [ ]:
def discord_hit_sheet(df_block):
    lines = ["NBA PROP ENGINE — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.FINAL_PROB*100:.1f}% | Edge: {r.FINAL_EDGE*100:+.1f} pts | MIN_L10: {r.MIN_L10:.1f}"
        )
    return "\n".join(lines)

print(discord_hit_sheet(final_top10))

In [ ]:
variance_mask = df["PROP"].str.contains("BLOCK|STEAL", case=False, na=False)
final_rank_novariance = df[(df["GAMES_FOUND"].fillna(0)>=3) & (~variance_mask)].copy()
final_rank_novariance = final_rank_novariance.sort_values("FINAL_PROB", ascending=False)
final_rank_novariance = final_rank_novariance.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10_novariance = final_rank_novariance.head(10).copy()
final_top10_novariance.insert(0, "RANK", range(1, len(final_top10_novariance)+1))

print(discord_hit_sheet(final_top10_novariance))

In [ ]:
import pandas as pd
import numpy as np
import re

CSV_PATH = "NBA 28 NIGHT.csv"  # uploaded file name

df = pd.read_csv(CSV_PATH)

# drop empty index column if present (common in exports)
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False)]
df.columns = [c.strip() for c in df.columns]

# normalize column names to match the notebook
rename_map = {
    "Player": "PLAYER",
    "IM PROB %": "IM_PROB_PCT",
    "HM/AW": "HM_AW",
    "DVP L5": "DVP_L5",
    "DVP L10": "DVP_L10",
    "DVP 2025": "DVP_2025",
    "DVP HM/AW": "DVP_HM_AW",
    "ODDS": "ODDS_RAW",
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

# parse AMERICAN_ODDS if embedded like "BET\n-169"
if "AMERICAN_ODDS" not in df.columns:
    if "ODDS_RAW" in df.columns:
        def extract_american(x):
            if pd.isna(x): return np.nan
            m = re.search(r"([+-]\d{2,5})", str(x))
            return float(m.group(1)) if m else np.nan
        df["AMERICAN_ODDS"] = df["ODDS_RAW"].apply(extract_american)
    else:
        df["AMERICAN_ODDS"] = np.nan

print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(5))

In [ ]:
import os

fname = "NBA 28 NIGHT.csv"
print("cwd:", os.getcwd())
print("exists:", os.path.exists(fname))
print("size bytes:", os.path.getsize(fname) if os.path.exists(fname) else None)
print("dir sample:", os.listdir(".")[:25])

In [ ]:
# ==============================
# NBA ENRICHMENTS (Pace + Defensive Rating) via NBA Stats API
# Adds: OPP_DEF_RTG, GAME_PACE
# ==============================

import pandas as pd
import numpy as np
import re
import requests

# Use FINAL_PROB if available, otherwise fallback to MODEL_PROB_ESPN_V3
PROB_COL = "FINAL_PROB" if "FINAL_PROB" in df.columns else "MODEL_PROB_ESPN_V3"
if PROB_COL not in df.columns:
    raise ValueError("Need FINAL_PROB or MODEL_PROB_ESPN_V3 in df. Run your base NBA model cells first.")

# Parse "AAA at BBB"
def parse_game_tokens(g):
    s = str(g).strip()
    m = re.match(r"^\s*([A-Z]{2,4})\s+at\s+([A-Z]{2,4})\s*$", s)
    if not m:
        return (None, None)
    return (m.group(1), m.group(2))

away_tokens, home_tokens = zip(*df["GAME"].map(parse_game_tokens))
df["AWAY_ABBR"] = away_tokens
df["HOME_ABBR"] = home_tokens

# NBA Stats headers
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "Accept-Language": "en-US,en;q=0.9",
}

def nba_stats_get(url, params):
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

# 1) Team list (abbr -> team_id)
teams_url = "https://stats.nba.com/stats/commonteamyears"
teams_js = nba_stats_get(teams_url, {"LeagueID": "00"})
teams_rs = teams_js["resultSets"][0]
teams_df = pd.DataFrame(teams_rs["rowSet"], columns=teams_rs["headers"])
abbr_map = teams_df.set_index("ABBREVIATION")["TEAM_ID"].to_dict()

# 2) Team advanced stats (PACE, DEF_RATING)
adv_url = "https://stats.nba.com/stats/leaguedashteamstats"
adv_js = nba_stats_get(adv_url, {
    "Conference": "",
    "DateFrom": "",
    "DateTo": "",
    "Division": "",
    "GameScope": "",
    "GameSegment": "",
    "LastNGames": "0",
    "LeagueID": "00",
    "Location": "",
    "MeasureType": "Advanced",
    "Month": "0",
    "OpponentTeamID": "0",
    "Outcome": "",
    "PORound": "0",
    "PaceAdjust": "N",
    "PerMode": "PerGame",
    "Period": "0",
    "PlayerExperience": "",
    "PlayerPosition": "",
    "PlusMinus": "N",
    "Rank": "N",
    "Season": "2025-26",     # adjust if needed
    "SeasonSegment": "",
    "SeasonType": "Regular Season",
    "ShotClockRange": "",
    "StarterBench": "",
    "TeamID": "0",
    "TwoWay": "0",
    "VsConference": "",
    "VsDivision": ""
})
adv_rs = adv_js["resultSets"][0]
adv_df = pd.DataFrame(adv_rs["rowSet"], columns=adv_rs["headers"])

team_adv = adv_df[["TEAM_ID","PACE","DEF_RATING"]].copy()
team_adv.rename(columns={"PACE":"TEAM_PACE","DEF_RATING":"TEAM_DEF_RTG"}, inplace=True)

# Map ids
df["HOME_ID"] = df["HOME_ABBR"].map(abbr_map)
df["AWAY_ID"] = df["AWAY_ABBR"].map(abbr_map)

# Merge team pace/def
df = df.merge(team_adv.add_prefix("HOME_"), left_on="HOME_ID", right_on="HOME_TEAM_ID", how="left")
df = df.merge(team_adv.add_prefix("AWAY_"), left_on="AWAY_ID", right_on="AWAY_TEAM_ID", how="left")

# Game pace proxy = average of both teams' pace
df["GAME_PACE"] = (df["HOME_TEAM_PACE"] + df["AWAY_TEAM_PACE"]) / 2.0

# Opponent for a prop row: if we can’t infer player team, use average opponent def (stable)
df["OPP_DEF_RTG"] = (df["HOME_TEAM_DEF_RTG"] + df["AWAY_TEAM_DEF_RTG"]) / 2.0

print("Merged NBA Pace/Def.")
print("Null GAME_PACE:", df["GAME_PACE"].isna().sum(), " / ", len(df))
print("Null OPP_DEF_RTG:", df["OPP_DEF_RTG"].isna().sum(), " / ", len(df))

ValueError: Need FINAL_PROB or MODEL_PROB_ESPN_V3 in df. Run your base NBA model cells first.

In [ ]:
# ==============================
# AUTO-DETECT HIT PROB COLUMN (NBA) + STANDARDIZE TO HIT_PROB
# Run this BEFORE the NBA enrichment cell
# ==============================

import numpy as np
import pandas as pd
import re

print("df shape:", df.shape)
print("df columns:", list(df.columns))

# Priority order (most likely -> least likely)
CANDIDATES = [
    "FINAL_PROB",
    "MODEL_PROB_ESPN_V3",
    "MODEL_PROB_CTX_V2",
    "MODEL_PROB_CTX",
    "MODEL_PROB",
    "PROB",
    "HIT_PROB",
    "P_HIT",
    "WIN_PROB",
]

prob_col = None
for c in CANDIDATES:
    if c in df.columns:
        prob_col = c
        break

# If none matched, try fuzzy find: any column that contains "prob" and looks numeric
if prob_col is None:
    prob_like = [c for c in df.columns if re.search(r"prob", str(c), re.I)]
    for c in prob_like:
        if pd.api.types.is_numeric_dtype(df[c]):
            prob_col = c
            break

# If still none, try hit-rate style columns (convert % -> prob if needed)
if prob_col is None:
    hit_like = [c for c in df.columns if re.search(r"hit|rate|pct|percent", str(c), re.I)]
    for c in hit_like:
        if pd.api.types.is_numeric_dtype(df[c]):
            prob_col = c
            break

if prob_col is None:
    raise ValueError(
        "Could not find a probability/hit-rate column in df.\n"
        "Run your base hit-rater/model cells first OR tell me which column is your probability."
    )

print("Using probability column:", prob_col)

# Standardize to HIT_PROB
df["HIT_PROB"] = df[prob_col].astype(float)

# If it looks like percentages (0–100), convert to 0–1
if df["HIT_PROB"].max() > 1.5:
    df["HIT_PROB"] = df["HIT_PROB"] / 100.0

df["HIT_PROB"] = df["HIT_PROB"].clip(0.01, 0.99)

print("HIT_PROB summary:")
display(df["HIT_PROB"].describe())

df shape: (101, 17)
df columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']


ValueError: Could not find a probability/hit-rate column in df.
Run your base hit-rater/model cells first OR tell me which column is your probability.

In [ ]:
# ==============================
# FIX NBA CSV LOAD (auto-detect header row) + rebuild df
# ==============================

import pandas as pd
import numpy as np
import os, re

CSV_PATH = "NBA 3:1 props.csv"   # if in Colab root; otherwise update to your path

# If you're unsure path, show files
print("Files in cwd:", [f for f in os.listdir(".") if f.lower().endswith(".csv")])

def find_header_row(path, max_rows=30):
    """
    Find the first row that looks like a header.
    We look for any of these tokens: player, game, prop, outcome, odds
    """
    preview = pd.read_csv(path, header=None, nrows=max_rows, dtype=str, engine="python")
    key_re = re.compile(r"(player|name|game|match|prop|outcome|odds|american)", re.I)
    best_i, best_score = None, -1
    for i in range(len(preview)):
        row = preview.iloc[i].fillna("").astype(str).tolist()
        score = sum(1 for x in row if key_re.search(x.strip()))
        if score > best_score:
            best_score = score
            best_i = i
    return best_i, best_score, preview.iloc[best_i].tolist()

hdr_i, score, hdr_row = find_header_row(CSV_PATH)
print("Detected header row index:", hdr_i, "score:", score)
print("Header row sample:", hdr_row[:12])

# Load using detected header row
df = pd.read_csv(CSV_PATH, header=hdr_i)

# Normalize column names
def norm_col(c):
    c = str(c).strip()
    c = re.sub(r"\s+", "_", c)
    c = c.replace("__", "_")
    return c

df.columns = [norm_col(c) for c in df.columns]

# Drop totally empty columns
df = df.loc[:, ~df.columns.str.match(r"^Unnamed", case=False)]
df = df.dropna(axis=1, how="all")

print("Loaded fixed df shape:", df.shape)
print("Columns:", list(df.columns))

# Quick checks for required fields (we'll also try to rename common variants)
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if lc in ["player", "player_name", "name"]: rename_map[c] = "PLAYER"
    if lc in ["game", "matchup", "match"]: rename_map[c] = "GAME"
    if "prop" in lc: rename_map[c] = "PROP"
    if lc in ["outcome", "side", "pick"]: rename_map[c] = "OUTCOME"
    if "american" in lc or lc in ["odds", "american_odds"]: rename_map[c] = "AMERICAN_ODDS"

df = df.rename(columns=rename_map)

print("After rename columns:", list(df.columns))

need = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]
missing = [c for c in need if c not in df.columns]
print("Missing required:", missing)

# Show top rows so you can confirm visually
display(df.head(5))

Files in cwd: ['NBA 3:1 props.csv']
Detected header row index: 1 score: 5
Header row sample: [nan, 'PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW']
Loaded fixed df shape: (100, 16)
Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM_PROB_%', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM/AW']
After rename columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'AMERICAN_ODDS', 'AVG', 'IM_PROB_%', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM/AW']
Missing required: []


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,AVG,IM_PROB_%,L5,L10,2025,HM/AW,H2H,DVP_L5,DVP_L10,DVP_2025,DVP_HM/AW
0,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,BET\n-110,-1102,52.4%,100%,100%,100%,100%,100%,100%,100%,100%,100%
1,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,BET\n+105,1052,48.8%,100%,100%,100%,100%,100%,100%,100%,100%,100%
2,Jarrett Allen,CLE at BKN,Rebounds,o0.5,BET\n-145,-1452,59.2%,100%,100%,100%,100%,100%,100%,100%,98.3%,96.6%
3,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,BET\n-129,-1304,56.3%,100%,90%,97.7%,100%,100%,20%,30%,15.5%,13.3%
4,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,BET\n-126,-1282,55.8%,100%,90%,95.3%,100%,100%,20%,30%,17.2%,16.7%


In [ ]:
# ==============================
# CLEAN NBA CSV (odds + % cols) + BUILD HIT_PROB (hit-rater model)
# ==============================

import numpy as np
import pandas as pd
import re

# --- 1) Fix AMERICAN_ODDS ---
def extract_american_odds(x):
    s = str(x)
    m = re.search(r"([+-]\d{2,4})", s.replace("−","-"))
    if m:
        return int(m.group(1))
    # sometimes odds got stuck like -1102 or 1052: take first 3-4 digits with sign
    m2 = re.search(r"([+-]\d{2,4})\d+", s.replace("−","-"))
    if m2:
        return int(m2.group(1))
    return np.nan

df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)

# Drop rows with no odds
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

# --- 2) Convert percent-like columns to floats in [0,1] ---
pct_cols = [c for c in df.columns if c in ["IM_PROB_%","L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]]

def pct_to_prob(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    s = s.replace("%","")
    s = s.replace("—","").replace("–","").replace("nan","")
    if s == "":
        return np.nan
    try:
        return float(s)/100.0
    except:
        return np.nan

for c in pct_cols:
    df[c] = df[c].apply(pct_to_prob)

# --- 3) Create BOOK_PROB from odds (market implied) ---
def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# --- 4) Build HIT_PROB from hit-rate features (symmetrical to NCAA) ---
# We’ll use a stable weighted blend with light DVP influence.
BASE_WEIGHTS = {
    "L5": 0.18,
    "L10": 0.22,
    "2025": 0.18,
    "HM/AW": 0.10,
    "H2H": 0.08,
    "DVP_L5": 0.08,
    "DVP_L10": 0.08,
    "DVP_2025": 0.06,
    "DVP_HM/AW": 0.02,
}

# Fill missing rates with column means (stable)
for k in BASE_WEIGHTS.keys():
    if k in df.columns:
        df[k] = df[k].astype(float)
        df[k] = df[k].fillna(df[k].mean())

# Weighted average
num = 0.0
den = 0.0
for k,w in BASE_WEIGHTS.items():
    if k in df.columns:
        num += w * df[k]
        den += w
df["HIT_PROB"] = (num/den).clip(0.01, 0.99)

# --- 5) Quick sanity prints ---
print("Rows:", len(df))
print("Odds sample:", df["AMERICAN_ODDS"].head(10).tolist())
print("HIT_PROB range:", (df["HIT_PROB"].min(), df["HIT_PROB"].max()))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","HIT_PROB"]].head(10))

Rows: 100
Odds sample: [-110, 105, -145, -129, -126, -125, -130, -142, -101, -115]
HIT_PROB range: (0.54434, 0.99)


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BOOK_PROB,HIT_PROB
0,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-110,0.523810,0.99000
1,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,105,0.487805,0.99000
2,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-145,0.591837,0.99000
3,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,-129,0.563319,0.78582
4,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.557522,0.78320
5,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u19.5,-125,0.555556,0.76760
6,Walter Clayton Jr.,MEM at IND,Points,u11.5,-130,0.565217,0.76184
7,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.586777,0.76676
8,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.502488,0.78552
9,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.534884,0.75420


In [ ]:
# ==============================
# NBA MATCHUP ENRICHMENT (Pace + Defensive Rating) via NBA Stats API
# ==============================

import pandas as pd
import numpy as np
import re
import requests

# Parse game tokens: supports "MIN at DEN" and "MIN @ DEN"
def parse_game_tokens(g):
    s = str(g).strip().upper()
    m = re.match(r"^\s*([A-Z]{2,4})\s+(?:AT|@)\s+([A-Z]{2,4})\s*$", s)
    return (m.group(1), m.group(2)) if m else (None, None)

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game_tokens))

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "Accept-Language": "en-US,en;q=0.9",
}

def nba_stats_get(url, params):
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

# Team list (abbr -> team_id)
teams_url = "https://stats.nba.com/stats/commonteamyears"
teams_js = nba_stats_get(teams_url, {"LeagueID": "00"})
teams_rs = teams_js["resultSets"][0]
teams_df = pd.DataFrame(teams_rs["rowSet"], columns=teams_rs["headers"])
abbr_map = teams_df.set_index("ABBREVIATION")["TEAM_ID"].to_dict()

# Team advanced stats (PACE, DEF_RATING)
adv_url = "https://stats.nba.com/stats/leaguedashteamstats"
adv_js = nba_stats_get(adv_url, {
    "LeagueID": "00",
    "Season": "2025-26",
    "SeasonType": "Regular Season",
    "PerMode": "PerGame",
    "MeasureType": "Advanced",
    "LastNGames": "0",
    "Month": "0",
    "OpponentTeamID": "0",
    "TeamID": "0",
    "PaceAdjust": "N",
    "PlusMinus": "N",
    "Rank": "N",
    "Conference": "",
    "Division": "",
    "Location": "",
    "Outcome": "",
    "PORound": "0",
    "Period": "0",
    "GameSegment": "",
    "DateFrom": "",
    "DateTo": "",
    "ShotClockRange": "",
    "PlayerExperience": "",
    "PlayerPosition": "",
    "StarterBench": "",
    "TwoWay": "0",
    "VsConference": "",
    "VsDivision": ""
})
adv_rs = adv_js["resultSets"][0]
adv_df = pd.DataFrame(adv_rs["rowSet"], columns=adv_rs["headers"])
team_adv = adv_df[["TEAM_ID","PACE","DEF_RATING"]].copy()
team_adv.rename(columns={"PACE":"TEAM_PACE","DEF_RATING":"TEAM_DEF_RTG"}, inplace=True)

# Map ids and merge
df["HOME_ID"] = df["HOME_ABBR"].map(abbr_map)
df["AWAY_ID"] = df["AWAY_ABBR"].map(abbr_map)

df = df.merge(team_adv.add_prefix("HOME_"), left_on="HOME_ID", right_on="HOME_TEAM_ID", how="left")
df = df.merge(team_adv.add_prefix("AWAY_"), left_on="AWAY_ID", right_on="AWAY_TEAM_ID", how="left")

df["GAME_PACE"] = (df["HOME_TEAM_PACE"] + df["AWAY_TEAM_PACE"]) / 2.0

# opponent DEF proxy (stable if we can't infer player team)
df["OPP_DEF_RTG"] = (df["HOME_TEAM_DEF_RTG"] + df["AWAY_TEAM_DEF_RTG"]) / 2.0

print("Merged NBA matchup context.")
print("Null GAME_PACE:", df["GAME_PACE"].isna().sum(), "/", len(df))
print("Null OPP_DEF_RTG:", df["OPP_DEF_RTG"].isna().sum(), "/", len(df))
display(df[["GAME","AWAY_ABBR","HOME_ABBR","GAME_PACE","OPP_DEF_RTG"]].drop_duplicates().head(10))

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
# ==============================
# MATCHUP PROXIES (NO NBA STATS CALLS)
# Uses DVP_* fields + HM/AW + season to create:
#   OPP_DEF_PROXY  (higher = tougher)
#   PACE_PROXY     (higher = faster)
# ==============================

import numpy as np
import pandas as pd

# Ensure DVP columns exist (they do in your CSV)
need = ["DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW","HM/AW","2025"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"Missing columns for proxies: {miss}")

# DVP proxies: treat as "opponent difficulty" signals; blend them
# (We only need a stable relative number to shift μ.)
df["OPP_DEF_PROXY_RAW"] = (
    0.30*df["DVP_L10"] +
    0.25*df["DVP_2025"] +
    0.20*df["DVP_L5"] +
    0.15*df["DVP_HM/AW"] +
    0.10*df["HM/AW"]
)

# Convert to a 0–100-ish scale where higher = tougher
# Use rank-based percentile to avoid weird units
df["OPP_DEF_PROXY"] = df["OPP_DEF_PROXY_RAW"].rank(pct=True) * 100.0

# Pace proxy: if you don't have a pace column, use a stable constant + minor spread from matchup percentile
# (Once NBA Stats works, this gets replaced by real pace.)
df["PACE_PROXY"] = 100.0 + (df["OPP_DEF_PROXY"].rank(pct=True) - 0.5) * 10.0  # 95–105 range

print("Proxy context built (no NBA Stats).")
display(df[["GAME","PLAYER","PROP","OPP_DEF_PROXY","PACE_PROXY"]].head(8))

Proxy context built (no NBA Stats).


,GAME,PLAYER,PROP,OPP_DEF_PROXY,PACE_PROXY
0,MIN at DEN,Anthony Edwards,Points + Rebounds + Assists,99.5,104.95
1,CLE at BKN,Michael Porter Jr.,Rebounds + Assists,99.5,104.95
2,CLE at BKN,Jarrett Allen,Rebounds,98.0,104.80
3,MIL at CHI,Rob Dillingham,Points + Rebounds + Assists,20.0,97.00
4,MIL at CHI,Rob Dillingham,Points + Assists,21.0,97.10
5,MEM at IND,Walter Clayton Jr.,Points + Rebounds + Assists,10.0,96.00
6,MEM at IND,Walter Clayton Jr.,Points,15.0,96.50
7,MEM at IND,Scotty Pippen Jr.,Rebounds,32.0,98.20


In [ ]:
# ==============================
# NBA FULL PIPELINE (NO NBA STATS)
# Uses:
#  HIT_PROB (from hit-rater blend cell)
#  OPP_DEF_PROXY + PACE_PROXY (from proxy cell)
# Outputs:
#  Elite 3–5 + Kelly units + clean Discord
# ==============================

import numpy as np
import pandas as pd
import re
from scipy.stats import norm, poisson, nbinom

# ---- helpers ----
def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

def dist_for_market(mkey):
    if mkey in ["three","steals","blocks"]:
        return "poisson"
    if mkey in ["assists","rebounds"]:
        return "nbinom"
    return "normal"

def nb_params_from_mu_k(mu, k):
    mu = max(0.05, float(mu))
    k = max(0.5, float(k))
    p = k/(k+mu)
    n = k
    return n, p

def poisson_prob_over(line, lam):
    k = int(np.floor(line))
    return float(1 - poisson.cdf(k, mu=lam))

def poisson_lam_from_pover(line, p_over):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    lo, hi = 0.05, max(12.0, line + 12.0)
    for _ in range(60):
        mid = (lo+hi)/2
        p = poisson_prob_over(line, mid)
        if p > p_over: hi = mid
        else: lo = mid
    return float((lo+hi)/2)

def nb_k_for_market(mkey):
    return 4.0 if mkey=="assists" else 5.0

def nb_prob_over(line, mu, k):
    n,p = nb_params_from_mu_k(mu, k)
    cut = int(np.floor(line))
    return float(1 - nbinom.cdf(cut, n=n, p=p))

def nb_mu_from_pover(line, p_over, k):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    lo, hi = 0.05, max(18.0, line + 18.0)
    for _ in range(60):
        mid = (lo+hi)/2
        p = nb_prob_over(line, mid, k)
        if p > p_over: hi = mid
        else: lo = mid
    return float((lo+hi)/2)

def normal_sigma(mkey, line):
    line = float(line)
    if mkey in ["pra","pr","pa","ra"]: return max(4.0, 0.17*line)
    if mkey == "points": return max(3.0, 0.18*line)
    return max(2.5, 0.18*max(line,1.0))

def normal_prob_over(line, mu, sigma):
    x = ((line + 0.5) - mu)/sigma
    return float(1 - norm.cdf(x))

def normal_mu_from_pover(line, p_over, sigma):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    z = norm.ppf(1 - p_over)
    return float((line + 0.5) - sigma*z)

def prob_from_mu(line, mu, dist_used, side, aux=np.nan, mkey="other"):
    if dist_used == "poisson":
        p_over = poisson_prob_over(line, mu)
    elif dist_used == "nbinom":
        k = float(aux) if pd.notna(aux) else nb_k_for_market(mkey)
        p_over = nb_prob_over(line, mu, k)
    else:
        sigma = float(aux) if pd.notna(aux) else normal_sigma(mkey, line)
        p_over = normal_prob_over(line, mu, sigma)
    return p_over if side=="over" else (1-p_over)

# ---- required columns ----
for c in ["HIT_PROB","BOOK_PROB","DEC_ODDS","OPP_DEF_PROXY","PACE_PROXY"]:
    if c not in df.columns:
        raise ValueError(f"Missing {c}. Run the prior cells first.")

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))
df["MKT"] = df["PROP"].map(market_key)

# ---- invert HIT_PROB -> μ_base ----
mu_base=[]; dist_used=[]; aux=[]
for r in df.itertuples(index=False):
    mkey = r.MKT
    dist = dist_for_market(mkey)
    dist_used.append(dist)

    p_over = r.HIT_PROB if r.SIDE=="over" else (1-r.HIT_PROB)
    line = float(r.LINE)

    if dist == "poisson":
        mu = poisson_lam_from_pover(line, p_over)
        mu_base.append(mu); aux.append(np.nan)
    elif dist == "nbinom":
        k = nb_k_for_market(mkey)
        mu = nb_mu_from_pover(line, p_over, k)
        mu_base.append(mu); aux.append(k)
    else:
        sigma = normal_sigma(mkey, line)
        mu = normal_mu_from_pover(line, p_over, sigma)
        mu_base.append(mu); aux.append(sigma)

df["DIST_USED"]=dist_used
df["AUX_PARAM"]=aux
df["PROJ_MU_BASE"]=mu_base

# ---- CTX-to-μ using proxies ----
BASE_DEF = float(df["OPP_DEF_PROXY"].median())   # ~50
BASE_PACE = float(df["PACE_PROXY"].median())     # ~100

# proxy coefficients
BETA = {
    "points": (0.55, 0.35),
    "rebounds": (0.20, 0.15),
    "assists": (0.18, 0.14),
    "three": (0.10, 0.08),
    "steals": (0.06, 0.05),
    "blocks": (0.06, 0.05),
    "pra": (0.85, 0.55),
    "pr": (0.70, 0.45),
    "pa": (0.70, 0.45),
    "ra": (0.55, 0.35),
    "other": (0.35, 0.25),
}
MU_CAP = {
    "points": 2.0, "rebounds": 1.2, "assists": 1.2, "three": 0.8,
    "steals": 0.5, "blocks": 0.5, "pra": 3.0, "pr": 2.5, "pa": 2.5, "ra": 2.0, "other": 1.5
}
def clip(v, lo, hi): return float(max(lo, min(hi, v)))

mu_ctx=[]; mu_shift=[]
for r in df.itertuples(index=False):
    mkey = r.MKT
    b_def, b_pace = BETA.get(mkey, BETA["other"])
    cap = MU_CAP.get(mkey, MU_CAP["other"])

    d_def = float(r.OPP_DEF_PROXY) - BASE_DEF      # +/- ~50
    d_pace = float(r.PACE_PROXY) - BASE_PACE       # +/- ~5

    # scale def by 25, pace by 5 (proxy units)
    shift = (b_def*(d_def/25.0)) + (b_pace*(d_pace/5.0))
    shift = clip(shift, -cap, cap)

    mu_shift.append(shift)
    mu_ctx.append(float(r.PROJ_MU_BASE) + shift)

df["PROJ_MU_SHIFT"]=mu_shift
df["PROJ_MU_CTX"]=mu_ctx

# ---- probability from μ_ctx ----
df["PROJ_PROB_CTX"] = [
    np.clip(prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT), 0.01, 0.99)
    for r in df.itertuples(index=False)
]

# ---- blend + Mode A shrink ----
W_HIT, W_PROJ = 0.70, 0.30
df["BLEND_PROB"] = np.clip(W_HIT*df["HIT_PROB"] + W_PROJ*df["PROJ_PROB_CTX"], 0.01, 0.99)

SHRINK_TO_BOOK = 0.25
df["SAFE_PROB"] = np.clip((1-SHRINK_TO_BOOK)*df["BLEND_PROB"] + SHRINK_TO_BOOK*df["BOOK_PROB"], 0.01, 0.99)
df["EV_SAFE_1U"] = df["SAFE_PROB"]*(df["DEC_ODDS"]-1) - (1-df["SAFE_PROB"])

# ---- injury sensitivity scenarios ----
UP_MULT  = {"points":1.06, "assists":1.07, "rebounds":1.04, "three":1.05, "pra":1.06, "pr":1.05, "pa":1.05, "ra":1.05, "other":1.05}
DOWN_MULT= {"points":0.92, "assists":0.90, "rebounds":0.93, "three":0.92, "pra":0.91, "pr":0.92, "pa":0.92, "ra":0.92, "other":0.92}

def apply_mu_scenario(mu, mkt, mode):
    if mode=="up": return float(mu)*UP_MULT.get(mkt,1.05)
    if mode=="down": return float(mu)*DOWN_MULT.get(mkt,0.92)
    return float(mu)

p_up=[]; p_dn=[]
for r in df.itertuples(index=False):
    mu0 = r.PROJ_MU_CTX
    pu = prob_from_mu(r.LINE, apply_mu_scenario(mu0, r.MKT, "up"), r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT)
    pdn= prob_from_mu(r.LINE, apply_mu_scenario(mu0, r.MKT, "down"), r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT)
    p_up.append(pu); p_dn.append(pdn)

df["INJ_SENS_PTS"] = (np.array(p_up) - np.array(p_dn)) * 100
df["INJ_SENS_ABS"] = df["INJ_SENS_PTS"].abs()
df["MU_SHIFT_ABS"] = df["PROJ_MU_SHIFT"].abs()

df["SAFE_SCORE"] = (df["EV_SAFE_1U"]*100) - (df["INJ_SENS_ABS"]*1.25) - (df["MU_SHIFT_ABS"]*6.0)

# ---- Elite 3–5 filter + Kelly units ----
MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_INJ  = 10.0

elite = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV) &
    (df["INJ_SENS_ABS"] <= MAX_INJ)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(5).copy()
elite["RANK"] = range(1, len(elite)+1)

HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f)*HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

elite["UNITS"] = [half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# ---- Clean Discord output ----
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in elite.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )
print("\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
1,1,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,105,0.37
0,2,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-110,0.37
2,3,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-145,0.36
38,4,Kobe Brown,MEM at IND,Assists,u1.5,130,0.25
8,5,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25


NBA ELITE PROP CARD
—
1) Michael Porter Jr. — CLE at BKN
Rebounds + Assists o0.5 (105) — 0.37u
2) Anthony Edwards — MIN at DEN
Points + Rebounds + Assists o0.5 (-110) — 0.37u
3) Jarrett Allen — CLE at BKN
Rebounds o0.5 (-145) — 0.36u
4) Kobe Brown — MEM at IND
Assists u1.5 (130) — 0.25u
5) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u


In [ ]:
# ==============================
# FIX BAD o0.5 LINES (NBA)
# - For PRA/RA/PR/PA/Reb/Ast/Pts: o0.5 is invalid -> rebuild line from AVG
# - Keep o0.5 ONLY for low-count markets if it makes sense (e.g., blocks/steals/threes)
# Then rerun elite extraction + Discord output
# ==============================

import numpy as np
import pandas as pd
import re

# 1) ensure AVG numeric
def to_float(x):
    if pd.isna(x): return np.nan
    s=str(x).strip().replace("%","")
    s=re.sub(r"[^0-9\.\-]", "", s)
    try: return float(s)
    except: return np.nan

df["AVG_NUM"] = df["AVG"].apply(to_float)

# 2) market key helper (must match pipeline)
def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

# 3) parse outcome safely
def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

# 4) identify bogus o0.5 for markets where it makes no sense
ALLOW_HALF_OVER = set(["three","steals","blocks"])   # can be o0.5 in these markets
bad = (
    (df["SIDE"]=="over") &
    (df["LINE"]==0.5) &
    (~df["MKT"].isin(ALLOW_HALF_OVER))
)

print("Bad o0.5 rows detected:", int(bad.sum()))
display(df.loc[bad, ["PLAYER","GAME","PROP","OUTCOME","AVG","AVG_NUM"]].head(10))

# 5) rebuild line from AVG for bad rows
# Use: line = round_to_half(AVG) with minimum reasonable floors by market
FLOOR = {
    "points": 6.5,
    "rebounds": 2.5,
    "assists": 1.5,
    "pra": 14.5,
    "pr": 10.5,
    "pa": 10.5,
    "ra": 6.5,
    "other": 4.5
}

def round_to_half(x):
    return np.round(x*2)/2

for idx in df.index[bad]:
    mkt = df.at[idx, "MKT"]
    avg = df.at[idx, "AVG_NUM"]
    if pd.isna(avg):
        # if no AVG, just drop later
        df.at[idx, "LINE"] = np.nan
        continue
    line = float(round_to_half(avg))
    line = max(line, FLOOR.get(mkt, 4.5))
    df.at[idx, "LINE"] = line
    df.at[idx, "OUTCOME"] = f"o{line:g}"  # keep over but correct line

# 6) drop any remaining nonsense (missing line)
pre = len(df)
df = df[df["LINE"].notna()].copy()
print("Dropped rows missing LINE:", pre - len(df))

# 7) rerun: recompute HIT_PROB / BOOK_PROB etc already exist; just rerun the elite pipeline parts
# (Assumes you've already run the pipeline cell that creates PROJ_MU_BASE etc; if not, rerun after this fix.)

print("Outcome/Line fixed. Sample:")
display(df[["PLAYER","PROP","OUTCOME","LINE","MKT","AVG_NUM"]].head(12))

Bad o0.5 rows detected: 4


,PLAYER,GAME,PROP,OUTCOME,AVG,AVG_NUM
0,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-1102,-1102.0
1,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,1052,1052.0
2,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-1452,-1452.0
14,Obi Toppin,MEM at IND,Assists,o0.5,-1963,-1963.0


Dropped rows missing LINE: 0
Outcome/Line fixed. Sample:


,PLAYER,PROP,OUTCOME,LINE,MKT,AVG_NUM
0,Anthony Edwards,Points + Rebounds + Assists,o14.5,14.5,pra,-1102.0
1,Michael Porter Jr.,Rebounds + Assists,o1052,1052.0,ra,1052.0
2,Jarrett Allen,Rebounds,o2.5,2.5,rebounds,-1452.0
3,Rob Dillingham,Points + Rebounds + Assists,u16.5,16.5,pra,-1304.0
4,Rob Dillingham,Points + Assists,u13.5,13.5,pa,-1282.0
5,Walter Clayton Jr.,Points + Rebounds + Assists,u19.5,19.5,pra,-1254.0
6,Walter Clayton Jr.,Points,u11.5,11.5,points,-1448.0
7,Scotty Pippen Jr.,Rebounds,u3.5,3.5,rebounds,-1839.0
8,Rob Dillingham,Points,u8.5,8.5,points,-11012.0
9,Walter Clayton Jr.,Points + Rebounds,u14.5,14.5,pr,-1225.0


In [ ]:
# ==============================
# FINAL FIX: DROP bogus o0.5 rows (non-count markets) + resanitize odds
# Then rerun your pipeline cell
# ==============================

import numpy as np
import pandas as pd
import re

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

def extract_american_odds(x):
    s = str(x)
    # prefer the first true odds token like -110 or +105
    m = re.search(r"([+-]\d{2,4})", s.replace("−","-"))
    if m:
        return int(m.group(1))
    # fallback: if it's numeric but too long, clip to 3-4 digits keeping sign
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})", s2)
    if m2:
        return int(m2.group(1) + m2.group(2))
    return np.nan

# Recompute market + line
df["MKT"] = df["PROP"].map(market_key)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

# Identify bogus 0.5 overs for non-count markets
ALLOW_HALF_OVER = set(["three","steals","blocks"])
bad = (df["SIDE"]=="over") & (df["LINE"]==0.5) & (~df["MKT"].isin(ALLOW_HALF_OVER))

print("Dropping bogus 0.5 rows:", int(bad.sum()))
display(df.loc[bad, ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]])

df = df.loc[~bad].copy()

# Re-sanitize odds (handles any -11012 type junk)
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

# Rebuild book probs / decimal odds
def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Post-fix rows:", len(df))
print("Odds sample:", df["AMERICAN_ODDS"].head(12).tolist())
print("Outcome sample:", df["OUTCOME"].head(12).tolist())

# IMPORTANT:
# Now rerun your long "NBA FULL PIPELINE" cell (μ + ctx + Mode A + Kelly + Discord)

Dropping bogus 0.5 rows: 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS


Post-fix rows: 85
Odds sample: [-110, -145, -129, -126, -125, -130, -142, -101, -115, -105, -110, -145]
Outcome sample: ['o14.5', 'o2.5', 'u16.5', 'u13.5', 'u19.5', 'u11.5', 'u3.5', 'u8.5', 'u14.5', 'u15.5', 'u18.5', 'u7.5']


In [ ]:
df.loc[df["LINE"] < 1, ["PLAYER","PROP","OUTCOME","AMERICAN_ODDS"]].head(20)

,PLAYER,PROP,OUTCOME,AMERICAN_ODDS
34,DeAndre Jordan,Assists,u0.5,-115
75,Tristan da Silva,Blocks,u0.5,-200
81,Dyson Daniels,Three Pointers Made,u0.5,-200


In [ ]:
# ==============================
# 0.5 LINE RULES (NBA)
# - Always allow 0.5 for three/steals/blocks (over or under)
# - For points/rebounds/assists/combos:
#     allow ONLY under 0.5
#     drop over 0.5
# ==============================

import numpy as np
import pandas as pd
import re

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["MKT"] = df["PROP"].map(market_key)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

half_ok_any = set(["three","steals","blocks"])
half_noncount = set(["points","rebounds","assists","pra","pr","pa","ra","other"])

drop_over_half = (
    (df["LINE"] == 0.5) &
    (df["SIDE"] == "over") &
    (df["MKT"].isin(half_noncount))
)

print("Dropping OVER 0.5 rows (non-count markets):", int(drop_over_half.sum()))
display(df.loc[drop_over_half, ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]].head(20))

df = df.loc[~drop_over_half].copy()

# Optional: if you want to exclude assists u0.5 entirely, uncomment:
# df = df[~((df["MKT"]=="assists") & (df["LINE"]==0.5))].copy()

print("Remaining 0.5 lines sample:")
display(df.loc[df["LINE"]==0.5, ["PLAYER","PROP","OUTCOME","AMERICAN_ODDS","MKT"]].head(20))

Dropping OVER 0.5 rows (non-count markets): 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS


Remaining 0.5 lines sample:


,PLAYER,PROP,OUTCOME,AMERICAN_ODDS,MKT
34,DeAndre Jordan,Assists,u0.5,-115,assists
75,Tristan da Silva,Blocks,u0.5,-200,blocks
81,Dyson Daniels,Three Pointers Made,u0.5,-200,three


In [ ]:
# ==============================
# MODE B: CLEAN 0.5 LINES
# - Keep 0.5 only for: three/blocks/steals
# - Drop 0.5 for assists/rebounds/points/combos
# ==============================

drop_half_noncount = (
    (df["LINE"] == 0.5) &
    (~df["MKT"].isin(["three","blocks","steals"]))
)

print("Dropping 0.5 lines outside count markets:", int(drop_half_noncount.sum()))
display(df.loc[drop_half_noncount, ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MKT"]].head(20))

df = df.loc[~drop_half_noncount].copy()

print("Remaining 0.5 lines:")
display(df.loc[df["LINE"]==0.5, ["PLAYER","PROP","OUTCOME","AMERICAN_ODDS","MKT"]].head(20))

Dropping 0.5 lines outside count markets: 1


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MKT
34,DeAndre Jordan,NOP at LAC,Assists,u0.5,-115,assists


Remaining 0.5 lines:


,PLAYER,PROP,OUTCOME,AMERICAN_ODDS,MKT
75,Tristan da Silva,Blocks,u0.5,-200,blocks
81,Dyson Daniels,Three Pointers Made,u0.5,-200,three


In [ ]:
# ==============================
# MODE B: ELITE CARD + KELLY + CLEAN DISCORD (no re-run needed)
# Requires df already has SAFE_PROB, EV_SAFE_1U, SAFE_SCORE, DEC_ODDS, AMERICAN_ODDS
# ==============================

import numpy as np

# If these aren't present, rerun the big pipeline cell once.
req = ["SAFE_PROB","EV_SAFE_1U","SAFE_SCORE","DEC_ODDS","AMERICAN_ODDS","PLAYER","GAME","PROP","OUTCOME"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns {missing}. Rerun the big pipeline cell once after Mode B filter.")

MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_PLAYS = 5

elite = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(MAX_PLAYS).copy()
elite["RANK"] = range(1, len(elite)+1)

# Half Kelly units (keep unsnapped units like NCAA)
HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f)*HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

elite["UNITS"] = [half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# Clean Discord output
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in elite.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )
print("\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,1,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,-110,0.37
2,2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,0.36
8,3,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25
21,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.25
4,5,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.25


NBA ELITE PROP CARD
—
1) Anthony Edwards — MIN at DEN
Points + Rebounds + Assists o14.5 (-110) — 0.37u
2) Jarrett Allen — CLE at BKN
Rebounds o2.5 (-145) — 0.36u
3) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u
4) Day'Ron Sharpe — CLE at BKN
Rebounds u9.5 (-116) — 0.25u
5) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.25u


In [ ]:
# ==============================
# TEMP EXCLUDE: Anthony Edwards PRA o14.5
# Then rebuild elite card
# ==============================

import numpy as np
import pandas as pd

# Remove that specific corrupted row
mask_exclude = (
    (df["PLAYER"].str.contains("Anthony Edwards", case=False, na=False)) &
    (df["PROP"].str.contains("Points + Rebounds + Assists", case=False, na=False)) &
    (df["LINE"] == 14.5)
)

print("Excluding rows:", int(mask_exclude.sum()))
display(df.loc[mask_exclude, ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]])

df = df.loc[~mask_exclude].copy()

# ---- rebuild elite card ----
MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_PLAYS = 5

elite = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(MAX_PLAYS).copy()
elite["RANK"] = range(1, len(elite)+1)

# Half Kelly
HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f)*HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

elite["UNITS"] = [half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# Clean Discord output
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in elite.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )

print("\n".join(lines))

Excluding rows: 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,1,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,-110,0.37
2,2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,0.36
8,3,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25
21,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.25
4,5,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.25


NBA ELITE PROP CARD
—
1) Anthony Edwards — MIN at DEN
Points + Rebounds + Assists o14.5 (-110) — 0.37u
2) Jarrett Allen — CLE at BKN
Rebounds o2.5 (-145) — 0.36u
3) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u
4) Day'Ron Sharpe — CLE at BKN
Rebounds u9.5 (-116) — 0.25u
5) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.25u


In [ ]:
# ==============================
# GUARANTEED EXCLUDE: Edwards PRA o14.5 (use OUTCOME text)
# ==============================

import pandas as pd

# 1) confirm the row exists in df
target = df[
    df["PLAYER"].str.contains("Anthony Edwards", case=False, na=False) &
    df["PROP"].str.contains("Points + Rebounds + Assists", case=False, na=False) &
    df["OUTCOME"].astype(str).str.strip().str.lower().eq("o14.5")
].copy()

print("Matches in df:", len(target))
display(target[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]].head(10))

# 2) drop it
df = df.drop(target.index).copy()

# 3) rebuild elite from scratch (no stale elite variable)
MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_PLAYS = 5

elite = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(MAX_PLAYS).copy()
elite["RANK"] = range(1, len(elite)+1)

# Half Kelly
HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f)*HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

elite["UNITS"] = [half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# Discord output
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in elite.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )
print("\n".join(lines))

Matches in df: 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,1,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,-110,0.37
2,2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,0.36
8,3,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25
21,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.25
4,5,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.25


NBA ELITE PROP CARD
—
1) Anthony Edwards — MIN at DEN
Points + Rebounds + Assists o14.5 (-110) — 0.37u
2) Jarrett Allen — CLE at BKN
Rebounds o2.5 (-145) — 0.36u
3) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u
4) Day'Ron Sharpe — CLE at BKN
Rebounds u9.5 (-116) — 0.25u
5) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.25u


In [ ]:
# ==============================
# Remove Edwards + Allen, then pull next two plays (old ranks 6 & 7)
# ==============================

import numpy as np

# 1) Build the full ranked board exactly like the last card selection
MIN_PROB = 0.60
MIN_EV   = 0.08

board = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV)
].copy()

board = board.sort_values("SAFE_SCORE", ascending=False).copy()
board["RANK_ALL"] = range(1, len(board)+1)

# 2) Snapshot the top 7 BEFORE removing anyone (so we can pull old ranks 6/7)
top7_before = board.head(7).copy()
print("Top 7 BEFORE removals:")
display(top7_before[["RANK_ALL","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","SAFE_PROB","EV_SAFE_1U","SAFE_SCORE"]])

# 3) Remove Edwards + Allen (robust contains match)
remove_mask = (
    top7_before["PLAYER"].str.contains("Anthony Edwards", case=False, na=False) |
    top7_before["PLAYER"].str.contains("Jarrett Allen", case=False, na=False)
)

# Old ranks 6 and 7 AFTER removing the two names:
# (i.e., take the top7 list and drop those names, then take the next two at the bottom)
remaining = top7_before.loc[~remove_mask].copy()

# If Edwards/Allen weren’t in the top7_before for some reason, fall back to removing from full board
if len(remaining) == len(top7_before):
    board_removed = board[
        ~(
            board["PLAYER"].str.contains("Anthony Edwards", case=False, na=False) |
            board["PLAYER"].str.contains("Jarrett Allen", case=False, na=False)
        )
    ].copy()
    # take top 5 from this cleaned board
    final = board_removed.head(5).copy()
else:
    # remaining currently contains (original top7 minus removed players) => should be 5 rows
    final = remaining.copy()
    final = final.head(5).copy()

# 4) Re-rank final list 1–5 and compute units (same half-Kelly)
final = final.copy()
final["RANK"] = range(1, len(final)+1)

HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f) * HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

final["UNITS"] = [half_kelly_units(p,d) for p,d in zip(final["SAFE_PROB"], final["DEC_ODDS"])]

print("\nFINAL CARD (Edwards + Allen removed):")
display(final[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# 5) Discord printout
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in final.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )
print("\n".join(lines))

# 6) Also explicitly show the two replacements (old ranks 6 & 7)
# These are the bottom 2 rows of the remaining top7_before after removals
replacements = remaining.tail(2).copy() if len(remaining) >= 5 else final.tail(2).copy()
print("\nREPLACEMENTS (the two you’re adding):")
display(replacements[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","SAFE_PROB","EV_SAFE_1U","SAFE_SCORE"]])

Top 7 BEFORE removals:


,RANK_ALL,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,SAFE_PROB,EV_SAFE_1U,SAFE_SCORE
0,1,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,-110,0.873452,0.667500,52.684133
2,2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,0.889146,0.502350,46.449934
8,3,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.724559,0.441945,32.394688
21,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.716886,0.334891,23.563812
4,5,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.742823,0.332364,17.957520
7,6,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.730011,0.244103,17.933711
9,7,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.721795,0.349443,16.551355



FINAL CARD (Edwards + Allen removed):


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
8,1,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25
21,2,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.25
4,3,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.25
7,4,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.25
9,5,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.25


NBA ELITE PROP CARD
—
1) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u
2) Day'Ron Sharpe — CLE at BKN
Rebounds u9.5 (-116) — 0.25u
3) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.25u
4) Scotty Pippen Jr. — MEM at IND
Rebounds u3.5 (-142) — 0.25u
5) Walter Clayton Jr. — MEM at IND
Points + Rebounds u14.5 (-115) — 0.25u

REPLACEMENTS (the two you’re adding):


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,SAFE_PROB,EV_SAFE_1U,SAFE_SCORE
7,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.730011,0.244103,17.933711
9,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.721795,0.349443,16.551355


In [ ]:
# ==============================
# STRICT NBA INPUT GATES (NCAA-symmetrical)
# - Fix odds parsing
# - Parse OUTCOME safely
# - Drop corrupted 0.5 overs in non-count markets
# - Drop insane line ranges by market
# ==============================

import numpy as np, pandas as pd, re

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    # numeric junk like -11012 -> take sign + first 3 digits
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

# sanitize odds
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

# parse outcome
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))
df["MKT"] = df["PROP"].map(market_key)

# Gate 1: drop OVER 0.5 for non-count markets
ALLOW_HALF_ANY = set(["three","steals","blocks"])
drop_over_half = (df["LINE"]==0.5) & (df["SIDE"]=="over") & (~df["MKT"].isin(ALLOW_HALF_ANY))
print("Dropping OVER 0.5 non-count:", int(drop_over_half.sum()))
df = df.loc[~drop_over_half].copy()

# Gate 2: line sanity by market (prevents Edwards PRA o14.5, Allen o2.5 junk, etc.)
RANGES = {
    "points":   (5.0, 45.0),
    "rebounds": (2.0, 20.0),
    "assists":  (1.0, 15.0),
    "three":    (0.5, 6.5),
    "blocks":   (0.5, 4.5),
    "steals":   (0.5, 4.5),
    "pra":      (20.0, 70.0),
    "pr":       (10.0, 55.0),
    "pa":       (10.0, 55.0),
    "ra":       (8.0, 35.0),
    "other":    (1.0, 80.0),
}
lo = df["MKT"].map(lambda k: RANGES.get(k, RANGES["other"])[0])
hi = df["MKT"].map(lambda k: RANGES.get(k, RANGES["other"])[1])
bad_line = (df["LINE"].isna()) | (df["LINE"] < lo) | (df["LINE"] > hi)

print("Dropping insane/missing lines:", int(bad_line.sum()))
display(df.loc[bad_line, ["PLAYER","GAME","PROP","OUTCOME","LINE","MKT","AMERICAN_ODDS"]].head(25))

df = df.loc[~bad_line].copy()

# rebuild book probs
def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("STRICT gates passed. Rows:", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MKT","LINE"]].head(10))

Dropping OVER 0.5 non-count: 0
Dropping insane/missing lines: 13


,PLAYER,GAME,PROP,OUTCOME,LINE,MKT,AMERICAN_ODDS
0,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,14.5,pra,-110
3,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,16.5,pra,-129
5,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u19.5,19.5,pra,-125
31,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u18.5,18.5,pra,-104
37,Rob Dillingham,MIL at CHI,Rebounds + Assists,u7.5,7.5,ra,-150
44,Kobe Brown,MEM at IND,Points + Rebounds + Assists,u19.5,19.5,pra,-105
47,Guerschon Yabusele,MIL at CHI,Rebounds + Assists,u7.5,7.5,ra,-109
48,Nolan Traore,CLE at BKN,Points + Rebounds + Assists,u19.5,19.5,pra,-125
52,DeMar DeRozan,SAC at LAL,Rebounds + Assists,o5.5,5.5,ra,-125
57,Nolan Traore,CLE at BKN,Points + Rebounds + Assists,u18.5,18.5,pra,-117


STRICT gates passed. Rows: 71


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MKT,LINE
2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,rebounds,2.5
4,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,pa,13.5
6,Walter Clayton Jr.,MEM at IND,Points,u11.5,-130,points,11.5
7,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,rebounds,3.5
8,Rob Dillingham,MIL at CHI,Points,u8.5,-101,points,8.5
9,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,pr,14.5
10,Walter Clayton Jr.,MEM at IND,Points + Assists,u15.5,-105,pa,15.5
11,Jarace Walker,MEM at IND,Points,u18.5,-110,points,18.5
12,Kobe Brown,MEM at IND,Rebounds,u7.5,-145,rebounds,7.5
13,Walter Clayton Jr.,MEM at IND,Three Pointers Made,u1.5,-142,three,1.5


In [ ]:
# ==============================
# ESPN ENRICHMENT (NBA) — DEF PPG ALLOWED + PACE PROXY
# - Pull team list + stats from ESPN
# - Cache results so it doesn't hammer endpoints
# ==============================

import requests, json, time
import pandas as pd
import numpy as np
import re

ESPN_TEAM_URL = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
ESPN_STATS_URL = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/statistics"

session = requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0"})

def get_json(url, tries=4, timeout=25):
    for i in range(tries):
        try:
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if i == tries-1:
                raise
            time.sleep(1.5*(i+1))
    return None

# Parse "AAA at BBB"
def parse_game_tokens(g):
    s = str(g).strip().upper()
    m = re.match(r"^\s*([A-Z]{2,4})\s+(?:AT|@)\s+([A-Z]{2,4})\s*$", s)
    return (m.group(1), m.group(2)) if m else (None, None)

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game_tokens))

# Team directory (abbr -> ESPN team id)
teams_js = get_json(ESPN_TEAM_URL)
teams = []
for item in teams_js.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
    t = item.get("team",{})
    teams.append({
        "abbr": t.get("abbreviation"),
        "team_id": t.get("id"),
        "name": t.get("displayName")
    })
team_dir = pd.DataFrame(teams).dropna()
abbr_to_id = dict(zip(team_dir["abbr"], team_dir["team_id"]))

df["HOME_ESPN_ID"] = df["HOME_ABBR"].map(abbr_to_id)
df["AWAY_ESPN_ID"] = df["AWAY_ABBR"].map(abbr_to_id)

need_ids = pd.unique(pd.concat([df["HOME_ESPN_ID"], df["AWAY_ESPN_ID"]]).dropna()).tolist()

# Cache stats
stats_cache = {}
def fetch_team_stats(team_id):
    if team_id in stats_cache:
        return stats_cache[team_id]
    js = get_json(ESPN_STATS_URL.format(team_id=team_id))
    # Find defensive PPG allowed and a pace-like proxy if available
    def_ppg = np.nan
    pace = np.nan

    # ESPN returns categories with stats; we search textually
    cats = js.get("categories", [])
    for c in cats:
        for st in c.get("stats", []):
            name = (st.get("name","") or "").lower()
            disp = (st.get("displayName","") or "").lower()
            val = st.get("value", None)

            # defensive points allowed per game (common fields)
            if "points allowed" in disp or "opp points" in disp or "opponent points" in disp:
                try: def_ppg = float(val)
                except: pass

            # pace / possessions proxy sometimes appears as "pace" or "possessions per game"
            if "pace" in disp or "possessions" in disp:
                try: pace = float(val)
                except: pass

    stats_cache[team_id] = {"DEF_PPG_ALLOWED": def_ppg, "PACE_PROXY": pace}
    return stats_cache[team_id]

rows=[]
for tid in need_ids:
    s = fetch_team_stats(tid)
    rows.append({"team_id": tid, **s})

ctx = pd.DataFrame(rows)

# Merge: opponent context (we don’t know player team reliably in this CSV)
# Use game-level proxies: avg of both teams
df = df.merge(ctx.add_prefix("HOME_"), left_on="HOME_ESPN_ID", right_on="HOME_team_id", how="left")
df = df.merge(ctx.add_prefix("AWAY_"), left_on="AWAY_ESPN_ID", right_on="AWAY_team_id", how="left")

df["ESPN_DEF_GAME"]  = (df["HOME_DEF_PPG_ALLOWED"] + df["AWAY_DEF_PPG_ALLOWED"]) / 2.0
df["ESPN_PACE_GAME"] = (df["HOME_PACE_PROXY"] + df["AWAY_PACE_PROXY"]) / 2.0

# If pace missing from ESPN (common), fallback to a stable median so we still run
df["ESPN_PACE_GAME"] = df["ESPN_PACE_GAME"].fillna(df["ESPN_PACE_GAME"].median())

print("ESPN enrichment merged.")
print("DEF null:", df["ESPN_DEF_GAME"].isna().sum(), "/", len(df))
print("PACE null:", df["ESPN_PACE_GAME"].isna().sum(), "/", len(df))
display(df[["GAME","ESPN_DEF_GAME","ESPN_PACE_GAME"]].drop_duplicates().head(10))

ESPN enrichment merged.
DEF null: 71 / 71
PACE null: 71 / 71


,GAME,ESPN_DEF_GAME,ESPN_PACE_GAME
0,CLE at BKN,NaN,NaN
1,MIL at CHI,NaN,NaN
2,MEM at IND,NaN,NaN
12,POR at ATL,NaN,NaN
17,PHI at BOS,NaN,NaN
24,DET at ORL,NaN,NaN
31,SAC at LAL,NaN,NaN
39,OKC at DAL,NaN,NaN
52,NOP at LAC,NaN,NaN


In [ ]:
# Replace proxy context with ESPN context (symmetrical to NCAA)
df["OPP_DEF_USED"] = df["ESPN_DEF_GAME"]
df["PACE_USED"]    = df["ESPN_PACE_GAME"]

BASE_DEF  = float(df["OPP_DEF_USED"].median())
BASE_PACE = float(df["PACE_USED"].median())

# then use OPP_DEF_USED and PACE_USED in your shift calculations
# d_def  = df["OPP_DEF_USED"] - BASE_DEF
# d_pace = df["PACE_USED"] - BASE_PACE

In [ ]:
import pandas as pd, numpy as np, re, os

CSV_PATH = "NBA 3:1 props.csv"  # your file in cwd

# detect header row (same method you used)
raw = pd.read_csv(CSV_PATH, header=None)
best_i, best_score = 0, -1
required = {"PLAYER","GAME","PROP","OUTCOME","ODDS"}
for i in range(min(10, len(raw))):
    row = set([str(x).strip().upper() for x in raw.iloc[i].tolist() if pd.notna(x)])
    score = sum([1 for r in required if r in row])
    if score > best_score:
        best_score = score
        best_i = i

hdr = raw.iloc[best_i].tolist()
df = raw.iloc[best_i+1:].copy()
df.columns = [str(c).strip() for c in hdr]
df = df.dropna(how="all")

# normalize expected names
rename_map = {
    "ODDS": "AMERICAN_ODDS",
    "IM PROB %": "IM_PROB_%",
    "IM PROB%": "IM_PROB_%",
}
df = df.rename(columns=rename_map)

# keep only known columns if present
keep = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","IM_PROB_%","AVG","L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
df = df[[c for c in keep if c in df.columns]].copy()

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Loaded df:", df.shape)
display(df.head(8))

Loaded df: (100, 16)


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,IM_PROB_%,AVG,L5,L10,2025,HM/AW,H2H,SIDE,LINE,BOOK_PROB,DEC_ODDS
2,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-110,52.4%,-1102,100%,100%,100%,100%,100%,over,0.5,0.523810,1.909091
3,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,105,48.8%,1052,100%,100%,100%,100%,100%,over,0.5,0.487805,2.050000
4,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-145,59.2%,-1452,100%,100%,100%,100%,100%,over,0.5,0.591837,1.689655
5,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,-129,56.3%,-1304,100%,90%,97.7%,100%,100%,under,16.5,0.563319,1.775194
6,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,55.8%,-1282,100%,90%,95.3%,100%,100%,under,13.5,0.557522,1.793651
7,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u19.5,-125,55.6%,-1254,100%,100%,90.2%,88%,100%,under,19.5,0.555556,1.800000
8,Walter Clayton Jr.,MEM at IND,Points,u11.5,-130,56.5%,-1448,100%,100%,86.3%,84%,100%,under,11.5,0.565217,1.769231
9,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,58.7%,-1839,100%,70%,100%,100%,100%,under,3.5,0.586777,1.704225


In [ ]:
import numpy as np, pandas as pd

def pct_to_prob(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    try:
        v = float(s)
        return v/100.0 if v > 1 else v
    except:
        return np.nan

# Convert all hit-rate columns to probabilities if present
hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob)

# Weighted blend (mirror NCAA style: recent + season + situational + DVP)
weights = {
    "L5":0.20, "L10":0.25, "2025":0.20, "HM/AW":0.10, "H2H":0.05,
    "DVP_L5":0.08, "DVP_L10":0.07, "DVP_2025":0.04, "DVP_HM/AW":0.01
}

num = 0.0
den = 0.0
for c,w in weights.items():
    if c in df.columns:
        v = df[c].astype(float)
        num += w * v.fillna(np.nan)
        den += w * v.notna().astype(float)

df["HIT_PROB"] = (num / den).clip(0.01, 0.99)

print("HIT_PROB null:", df["HIT_PROB"].isna().sum(), "/", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","HIT_PROB"]].head(10))

HIT_PROB null: 2 / 100


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,HIT_PROB
2,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-110,0.99000
3,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,105,0.99000
4,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-145,0.99000
5,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,-129,0.96300
6,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.95700
7,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u19.5,-125,0.96050
8,Walter Clayton Jr.,MEM at IND,Points,u11.5,-130,0.94575
9,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.90625
10,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.91475
11,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.94075


In [ ]:
import requests, time, re
import pandas as pd, numpy as np

session = requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0"})

def get_json(url, tries=4, timeout=25):
    for i in range(tries):
        try:
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception:
            if i == tries-1:
                raise
            time.sleep(1.5*(i+1))

ESPN_TEAM_URL  = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
ESPN_STATS_URL = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/statistics"

def parse_game_tokens(g):
    s = str(g).strip().upper()
    m = re.match(r"^\s*([A-Z]{2,4})\s+(?:AT|@)\s+([A-Z]{2,4})\s*$", s)
    return (m.group(1), m.group(2)) if m else (None, None)

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game_tokens))

teams_js = get_json(ESPN_TEAM_URL)
teams = []
for item in teams_js.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
    t = item.get("team",{})
    teams.append({"abbr": t.get("abbreviation"), "team_id": t.get("id")})
team_dir = pd.DataFrame(teams).dropna()
abbr_to_id = dict(zip(team_dir["abbr"], team_dir["team_id"]))

df["HOME_ESPN_ID"] = df["HOME_ABBR"].map(abbr_to_id)
df["AWAY_ESPN_ID"] = df["AWAY_ABBR"].map(abbr_to_id)

need_ids = pd.unique(pd.concat([df["HOME_ESPN_ID"], df["AWAY_ESPN_ID"]]).dropna()).tolist()

stats_cache = {}
def fetch_team_stats(team_id):
    if team_id in stats_cache:
        return stats_cache[team_id]
    js = get_json(ESPN_STATS_URL.format(team_id=team_id))
    def_ppg = np.nan
    pace = np.nan
    cats = js.get("categories", [])
    for c in cats:
        for st in c.get("stats", []):
            disp = (st.get("displayName","") or "").lower()
            val  = st.get("value", None)
            if ("points allowed" in disp) or ("opp points" in disp) or ("opponent points" in disp):
                try: def_ppg = float(val)
                except: pass
            if ("pace" in disp) or ("possessions" in disp):
                try: pace = float(val)
                except: pass
    stats_cache[team_id] = {"DEF_PPG_ALLOWED": def_ppg, "PACE_PROXY": pace}
    return stats_cache[team_id]

rows=[]
for tid in need_ids:
    s = fetch_team_stats(tid)
    rows.append({"team_id": tid, **s})
ctx = pd.DataFrame(rows)

df = df.merge(ctx.add_prefix("HOME_"), left_on="HOME_ESPN_ID", right_on="HOME_team_id", how="left")
df = df.merge(ctx.add_prefix("AWAY_"), left_on="AWAY_ESPN_ID", right_on="AWAY_team_id", how="left")

df["ESPN_DEF_GAME"]  = (df["HOME_DEF_PPG_ALLOWED"] + df["AWAY_DEF_PPG_ALLOWED"]) / 2.0
df["ESPN_PACE_GAME"] = (df["HOME_PACE_PROXY"] + df["AWAY_PACE_PROXY"]) / 2.0

# pace often missing -> fallback to median
df["ESPN_PACE_GAME"] = df["ESPN_PACE_GAME"].fillna(df["ESPN_PACE_GAME"].median())

print("ESPN DEF null:", df["ESPN_DEF_GAME"].isna().sum(), "/", len(df))
print("ESPN PACE null:", df["ESPN_PACE_GAME"].isna().sum(), "/", len(df))
display(df[["GAME","ESPN_DEF_GAME","ESPN_PACE_GAME"]].drop_duplicates().head(10))

ESPN DEF null: 100 / 100
ESPN PACE null: 100 / 100


,GAME,ESPN_DEF_GAME,ESPN_PACE_GAME
0,MIN at DEN,NaN,NaN
1,CLE at BKN,NaN,NaN
3,MIL at CHI,NaN,NaN
5,MEM at IND,NaN,NaN
16,POR at ATL,NaN,NaN
22,PHI at BOS,NaN,NaN
32,DET at ORL,NaN,NaN
34,NOP at LAC,NaN,NaN
42,SAC at LAL,NaN,NaN
54,OKC at DAL,NaN,NaN


In [ ]:
import numpy as np, pandas as pd, re
from scipy.stats import norm, poisson, nbinom

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

# STRICT gate: drop insane lines by market (prevents PRA o14.5 ever)
RANGES = {
    "points": (5.0,45.0), "rebounds":(2.0,20.0), "assists":(1.0,15.0),
    "three":(0.5,6.5), "blocks":(0.5,4.5), "steals":(0.5,4.5),
    "pra":(20.0,70.0), "pr":(10.0,55.0), "pa":(10.0,55.0), "ra":(8.0,35.0),
    "other":(1.0,80.0)
}
lo = df["MKT"].map(lambda k: RANGES.get(k, RANGES["other"])[0])
hi = df["MKT"].map(lambda k: RANGES.get(k, RANGES["other"])[1])
df = df[(df["LINE"].notna()) & (df["LINE"]>=lo) & (df["LINE"]<=hi)].copy()

# Distributions
def dist_for_market(mkey):
    if mkey in ["three","steals","blocks"]:
        return "poisson"
    if mkey in ["assists","rebounds"]:
        return "nbinom"
    return "normal"

def nb_params_from_mu_k(mu, k):
    mu = max(0.05, float(mu)); k = max(0.5, float(k))
    p = k/(k+mu); n = k
    return n,p

def poisson_prob_over(line, lam):
    cut = int(np.floor(line))
    return float(1 - poisson.cdf(cut, mu=lam))

def poisson_lam_from_pover(line, p_over):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    lo, hi = 0.05, max(12.0, line+12.0)
    for _ in range(60):
        mid=(lo+hi)/2
        p=poisson_prob_over(line, mid)
        if p>p_over: hi=mid
        else: lo=mid
    return float((lo+hi)/2)

def nb_k_for_market(mkey): return 4.0 if mkey=="assists" else 5.0

def nb_prob_over(line, mu, k):
    n,p=nb_params_from_mu_k(mu,k)
    cut=int(np.floor(line))
    return float(1-nbinom.cdf(cut, n=n, p=p))

def nb_mu_from_pover(line, p_over, k):
    p_over=float(np.clip(p_over,0.02,0.98))
    lo,hi=0.05,max(18.0,line+18.0)
    for _ in range(60):
        mid=(lo+hi)/2
        p=nb_prob_over(line, mid, k)
        if p>p_over: hi=mid
        else: lo=mid
    return float((lo+hi)/2)

def normal_sigma(mkey, line):
    line=float(line)
    if mkey in ["pra","pr","pa","ra"]: return max(4.0, 0.17*line)
    if mkey=="points": return max(3.0, 0.18*line)
    return max(2.5, 0.18*max(line,1.0))

def normal_prob_over(line, mu, sigma):
    x=((line+0.5)-mu)/sigma
    return float(1-norm.cdf(x))

def normal_mu_from_pover(line, p_over, sigma):
    p_over=float(np.clip(p_over,0.02,0.98))
    z=norm.ppf(1-p_over)
    return float((line+0.5) - sigma*z)

def prob_from_mu(line, mu, dist_used, side, aux=np.nan, mkey="other"):
    if dist_used=="poisson":
        p_over=poisson_prob_over(line, mu)
    elif dist_used=="nbinom":
        k=float(aux) if pd.notna(aux) else nb_k_for_market(mkey)
        p_over=nb_prob_over(line, mu, k)
    else:
        sigma=float(aux) if pd.notna(aux) else normal_sigma(mkey, line)
        p_over=normal_prob_over(line, mu, sigma)
    return p_over if side=="over" else (1-p_over)

# invert HIT_PROB -> μ_base
dist_used=[]; aux=[]; mu_base=[]
for r in df.itertuples(index=False):
    dist=dist_for_market(r.MKT)
    dist_used.append(dist)
    p_over = r.HIT_PROB if r.SIDE=="over" else (1-r.HIT_PROB)
    if dist=="poisson":
        mu=poisson_lam_from_pover(r.LINE, p_over); mu_base.append(mu); aux.append(np.nan)
    elif dist=="nbinom":
        k=nb_k_for_market(r.MKT)
        mu=nb_mu_from_pover(r.LINE, p_over, k); mu_base.append(mu); aux.append(k)
    else:
        sig=normal_sigma(r.MKT, r.LINE)
        mu=normal_mu_from_pover(r.LINE, p_over, sig); mu_base.append(mu); aux.append(sig)

df["DIST_USED"]=dist_used
df["AUX_PARAM"]=aux
df["PROJ_MU_BASE"]=mu_base

# CTX-to-μ using ESPN DEF/PACE (like NCAA today)
df["OPP_DEF_USED"] = df["ESPN_DEF_GAME"]
df["PACE_USED"]    = df["ESPN_PACE_GAME"]
BASE_DEF  = float(df["OPP_DEF_USED"].median())
BASE_PACE = float(df["PACE_USED"].median())

BETA = {
    "points": (0.55, 0.35), "rebounds": (0.20, 0.15), "assists": (0.18, 0.14),
    "three": (0.10, 0.08), "steals": (0.06, 0.05), "blocks": (0.06, 0.05),
    "pra": (0.85, 0.55), "pr": (0.70, 0.45), "pa": (0.70, 0.45), "ra": (0.55, 0.35),
    "other": (0.35, 0.25),
}
MU_CAP = {"points":2.0,"rebounds":1.2,"assists":1.2,"three":0.8,"steals":0.5,"blocks":0.5,"pra":3.0,"pr":2.5,"pa":2.5,"ra":2.0,"other":1.5}

def clip(v, lo_, hi_):
    return float(max(lo_, min(hi_, v)))

mu_shift=[]; mu_ctx=[]
for r in df.itertuples(index=False):
    b_def,b_pace = BETA.get(r.MKT, BETA["other"])
    cap = MU_CAP.get(r.MKT, MU_CAP["other"])
    d_def  = (float(r.OPP_DEF_USED) - BASE_DEF) / 8.0
    d_pace = (float(r.PACE_USED) - BASE_PACE) / 2.5
    shift = (b_def*d_def) + (b_pace*d_pace)
    shift = clip(shift, -cap, cap)
    mu_shift.append(shift)
    mu_ctx.append(float(r.PROJ_MU_BASE) + shift)

df["PROJ_MU_SHIFT"]=mu_shift
df["PROJ_MU_CTX"]=mu_ctx

df["PROJ_PROB_CTX"] = [
    np.clip(prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT), 0.01, 0.99)
    for r in df.itertuples(index=False)
]

# Blend (same as NCAA today)
W_HIT, W_PROJ = 0.70, 0.30
df["BLEND_PROB"] = np.clip(W_HIT*df["HIT_PROB"] + W_PROJ*df["PROJ_PROB_CTX"], 0.01, 0.99)

# Mode A shrink toward book
SHRINK_TO_BOOK = 0.25
df["SAFE_PROB"] = np.clip((1-SHRINK_TO_BOOK)*df["BLEND_PROB"] + SHRINK_TO_BOOK*df["BOOK_PROB"], 0.01, 0.99)
df["EV_SAFE_1U"] = df["SAFE_PROB"]*(df["DEC_ODDS"]-1) - (1-df["SAFE_PROB"])

# Injury sensitivity (scenario swing)
UP_MULT  = {"points":1.06,"assists":1.07,"rebounds":1.04,"three":1.05,"pra":1.06,"pr":1.05,"pa":1.05,"ra":1.05,"other":1.05,"steals":1.04,"blocks":1.04}
DOWN_MULT= {"points":0.92,"assists":0.90,"rebounds":0.93,"three":0.92,"pra":0.91,"pr":0.92,"pa":0.92,"ra":0.92,"other":0.92,"steals":0.94,"blocks":0.94}

def apply_mu(mu, mkt, mode):
    if mode=="up": return float(mu)*UP_MULT.get(mkt,1.05)
    return float(mu)*DOWN_MULT.get(mkt,0.92)

p_up=[]; p_dn=[]
for r in df.itertuples(index=False):
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX, r.MKT, "up"), r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX, r.MKT, "down"), r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT))

delta = np.clip(np.array(p_up) - np.array(p_dn), -1, 1)
df["INJ_SENS_ABS"] = np.abs(delta) * 100
df["MU_SHIFT_ABS"] = df["PROJ_MU_SHIFT"].abs()

df["SAFE_SCORE"] = (df["EV_SAFE_1U"]*100) - (df["INJ_SENS_ABS"]*1.25) - (df["MU_SHIFT_ABS"]*6.0)

# Elite selection + half Kelly
MIN_PROB=0.60; MIN_EV=0.08; MAX_PLAYS=5
elite = df[(df["SAFE_PROB"]>=MIN_PROB) & (df["EV_SAFE_1U"]>=MIN_EV)].copy()
elite = elite.sort_values("SAFE_SCORE", ascending=False).head(MAX_PLAYS).copy()
elite["RANK"]=range(1,len(elite)+1)

HALF_KELLY=0.50; MIN_U=0.25; MAX_U=1.00
def half_kelly_units(p, dec):
    b=dec-1; q=1-p
    f=(b*p-q)/b
    f=max(0.0,f)*HALF_KELLY
    u=min(max(f,0.0),MAX_U)
    if 0<u<MIN_U: u=MIN_U
    return float(round(u,2))

elite["UNITS"]=[half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

lines=["NBA ELITE PROP CARD","—"]
for r in elite.itertuples(index=False):
    lines.append(f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)}) — {r.UNITS:.2f}u")
print("\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
8,1,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.28
4,2,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.31
9,3,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.30
60,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u8.5,113,0.25
38,5,Kobe Brown,MEM at IND,Assists,u1.5,130,0.25


NBA ELITE PROP CARD
—
1) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.28u
2) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.31u
3) Walter Clayton Jr. — MEM at IND
Points + Rebounds u14.5 (-115) — 0.30u
4) Day'Ron Sharpe — CLE at BKN
Rebounds u8.5 (113) — 0.25u
5) Kobe Brown — MEM at IND
Assists u1.5 (130) — 0.25u


In [ ]:
import pandas as pd, numpy as np, re

CSV_PATH = "/mnt/data/NBA prop 3:2.csv"

# Detect header row (robust for messy exports)
raw = pd.read_csv(CSV_PATH, header=None)
best_i, best_score = 0, -1
required = {"PLAYER","GAME","PROP","OUTCOME","ODDS"}
for i in range(min(15, len(raw))):
    row = set([str(x).strip().upper() for x in raw.iloc[i].tolist() if pd.notna(x)])
    score = sum([1 for r in required if r in row])
    if score > best_score:
        best_score = score
        best_i = i

hdr = raw.iloc[best_i].tolist()
df = raw.iloc[best_i+1:].copy()
df.columns = [str(c).strip() for c in hdr]
df = df.dropna(how="all").copy()

# Normalize expected names
rename_map = {
    "ODDS": "AMERICAN_ODDS",
    "IM PROB %": "IM_PROB_%",
    "IM PROB%": "IM_PROB_%",
}
df = df.rename(columns=rename_map)

# Keep only known columns if present
keep = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","IM_PROB_%","AVG","L5","L10","2025","HM/AW","H2H",
        "DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
df = df[[c for c in keep if c in df.columns]].copy()

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Loaded:", CSV_PATH)
print("df shape:", df.shape)
print("columns:", list(df.columns))
display(df.head(8))

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/NBA prop 3:2.csv'

CSV candidates found: []


FileNotFoundError: No CSV found. Upload NBA prop 3:2.csv to this Colab runtime (or mount Drive).

In [ ]:
from google.colab import files
uploaded = files.upload()

# Grab uploaded filename automatically
import os
CSV_PATH = list(uploaded.keys())[0]

print("Uploaded file:", CSV_PATH)


Saving NBA prop 3:2.csv to NBA prop 3:2 (1).csv
Uploaded file: NBA prop 3:2 (1).csv


In [ ]:
import pandas as pd, numpy as np, re, os

print("Using file:", CSV_PATH)

raw = pd.read_csv(CSV_PATH, header=None)

# -------- STEP 1: find real header row --------
header_index = None

for i in range(min(40, len(raw))):
    row_vals = [str(x).strip().upper() for x in raw.iloc[i].tolist()]
    if "PLAYER" in row_vals and "GAME" in row_vals and "PROP" in row_vals:
        header_index = i
        break

if header_index is None:
    raise Exception("Could not find header row containing PLAYER/GAME/PROP")

print("Detected header row index:", header_index)

# -------- STEP 2: rebuild df using real header --------
headers = [str(x).strip() for x in raw.iloc[header_index].tolist()]
df = raw.iloc[header_index+1:].copy()
df.columns = headers
df = df.dropna(how="all").copy()

print("Columns detected:", df.columns.tolist())

# -------- STEP 3: detect odds column --------
odds_col = None
for c in df.columns:
    sample = df[c].astype(str).head(30)
    hits = sample.str.contains(r"[+-]\d{2,4}", regex=True).sum()
    if hits > 5:  # strong signal of odds column
        odds_col = c
        break

if odds_col is None:
    raise Exception("Could not auto-detect odds column")

print("Detected odds column:", odds_col)

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    return int(m.group(1)) if m else np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df[odds_col].apply(extract_american_odds)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

def american_to_prob(o):
    return (-o)/((-o)+100) if o < 0 else 100/(o+100)

def american_to_decimal(o):
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"] = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Rows loaded:", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","LINE","SIDE"]].head(10))

Using file: NBA prop 3:2 (1).csv


Exception: Could not find header row containing PLAYER/GAME/PROP

In [ ]:
import pandas as pd, numpy as np, re, os

print("Using file:", CSV_PATH)

raw = pd.read_csv(CSV_PATH, header=None, dtype=str, keep_default_na=False)

# --- Header row detector: score each row by "header-likeness"
TOKENS = [
    "PLAYER","NAME","GAME","MATCHUP","PROP","MARKET","OUTCOME","O/U","LINE",
    "ODDS","PRICE","BET","BOOK","IM PROB","PROB","L5","L10","H2H","DVP","2025","HM","AW"
]

def row_score(row):
    vals = [str(x).strip().upper() for x in row]
    # remove empties
    vals = [v for v in vals if v not in ["", "NAN", "NONE"]]
    if not vals:
        return -999
    joined = " | ".join(vals)
    score = 0
    for t in TOKENS:
        if t in joined:
            score += 1
    # bonus if row has lots of distinct text labels (typical headers)
    alpha_cells = sum(1 for v in vals if re.search(r"[A-Z]", v))
    score += min(alpha_cells, 10) * 0.2
    return score

scores = []
for i in range(min(80, len(raw))):
    scores.append((i, row_score(raw.iloc[i].tolist())))

header_index, best = max(scores, key=lambda x: x[1])
print("Best header row index:", header_index, "score:", best)

headers = [str(x).strip() for x in raw.iloc[header_index].tolist()]
# If there are blanks, fill them with placeholders so pandas doesn't merge columns into DataFrames
headers = [h if h not in ["", "nan", "NaN", "None", None] else f"COL_{j}" for j,h in enumerate(headers)]

df = raw.iloc[header_index+1:].copy()
df.columns = headers
df = df.replace({"": np.nan}).dropna(how="all").copy()

print("Detected header sample:", headers[:20])
print("df shape:", df.shape)

# --- Auto-detect ODDS column by counting +105/-110 patterns
def odds_hits(series, n=60):
    s = series.astype(str).head(min(n, len(series)))
    return s.str.contains(r"([+-]\d{2,4})", regex=True).sum()

best_col, best_hits = None, -1
for c in df.columns:
    h = odds_hits(df[c])
    if h > best_hits:
        best_hits = h
        best_col = c

if best_hits < 3:
    # fallback: look for a column name with odds-like words
    name_hits = [c for c in df.columns if any(k in str(c).upper() for k in ["ODDS","PRICE","BET","LINE"])]
    best_col = name_hits[0] if name_hits else best_col

if best_col is None:
    raise Exception("Could not detect odds column in this CSV export.")

ODDS_COL = best_col
print("Detected ODDS_COL =", ODDS_COL, "pattern hits:", best_hits)

# --- Parse odds + outcome line
def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})$", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

# Find the OUTCOME column (it may not be literally called OUTCOME)
outcome_candidates = [c for c in df.columns if str(c).strip().upper() in ["OUTCOME","O/U","OU","TOTAL","SIDE"]]
if not outcome_candidates:
    # heuristic: pick a column where values look like o12.5 / u8.5
    best_o_col, best_o_hits = None, -1
    for c in df.columns:
        s = df[c].astype(str).head(80)
        hits = s.str.contains(r"^(o|u)\s*\d+(\.\d+)?$", case=False, regex=True).sum()
        if hits > best_o_hits:
            best_o_hits = hits
            best_o_col = c
    outcome_candidates = [best_o_col] if best_o_col is not None else []

if not outcome_candidates or outcome_candidates[0] is None:
    raise Exception("Could not detect OUTCOME column. This export is missing o/u lines.")

OUTCOME_COL = outcome_candidates[0]
print("Detected OUTCOME_COL =", OUTCOME_COL)

# Find PLAYER/GAME/PROP columns (again: not always exact names)
def pick_col(name_keywords):
    candidates = [c for c in df.columns if any(k in str(c).upper() for k in name_keywords)]
    return candidates[0] if candidates else None

PLAYER_COL = pick_col(["PLAYER","NAME"])
GAME_COL   = pick_col(["GAME","MATCHUP"])
PROP_COL   = pick_col(["PROP","MARKET"])

if PLAYER_COL is None or GAME_COL is None or PROP_COL is None:
    print("WARNING: Could not perfectly detect PLAYER/GAME/PROP column names.")
    print("Detected:", {"PLAYER_COL":PLAYER_COL,"GAME_COL":GAME_COL,"PROP_COL":PROP_COL})

# Normalize to expected names
if PLAYER_COL: df = df.rename(columns={PLAYER_COL:"PLAYER"})
if GAME_COL:   df = df.rename(columns={GAME_COL:"GAME"})
if PROP_COL:   df = df.rename(columns={PROP_COL:"PROP"})
df = df.rename(columns={OUTCOME_COL:"OUTCOME"})

df["AMERICAN_ODDS"] = df[ODDS_COL].apply(extract_american_odds)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Final df shape:", df.shape)
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","LINE","SIDE"]].head(12))

Using file: NBA prop 3:2 (1).csv
Best header row index: 7 score: 3.0
Detected header sample: ['COL_0', 'Julian Strawther', 'DEN at UTA', 'Rebounds', 'u3.5', 'BET\n-137', '-1455', '57.8%', '80%', '60%', '78.4%', '70.6%', '100%', '0%', '20%', '33.3%', '32.3%']
df shape: (93, 17)
Detected ODDS_COL = BET
-137 pattern hits: 60
Detected OUTCOME_COL = u3.5
Detected: {'PLAYER_COL': None, 'GAME_COL': None, 'PROP_COL': None}
Final df shape: (93, 22)


/tmp/ipython-input-1748/2232817705.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({"": np.nan}).dropna(how="all").copy()
/tmp/ipython-input-1748/2232817705.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return s.str.contains(r"([+-]\d{2,4})", regex=True).sum()
/tmp/ipython-input-1748/2232817705.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return s.str.contains(r"([+-]\d{2,4})", regex=True).sum()
/tmp/ipython-input-1748/2232817705.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the group

KeyError: "['PLAYER', 'GAME', 'PROP'] not in index"

In [ ]:
import pandas as pd, numpy as np, re

CSV_PATH = "NBA prop 3:2.csv"   # or whatever your uploaded filename is

raw = pd.read_csv(CSV_PATH, header=None, dtype=str, keep_default_na=False)

# Drop fully blank rows
raw = raw.replace("", np.nan).dropna(how="all").fillna("")

# Your file is "headerless" with a blank first column. Map by position.
# 0 blank, 1 player, 2 game, 3 prop, 4 outcome, 5 odds, 6 junk/extra, 7 im prob, 8..16 hit columns
colmap = {
    1: "PLAYER",
    2: "GAME",
    3: "PROP",
    4: "OUTCOME",
    5: "ODDS_RAW",
    7: "IM_PROB_%",
    8: "L5",
    9: "L10",
    10: "2025",
    11: "HM/AW",
    12: "H2H",
    13: "DVP_L5",
    14: "DVP_L10",
    15: "DVP_2025",
    16: "DVP_HM/AW",
}

df = raw.rename(columns=colmap)[list(colmap.values())].copy()

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})$", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df["ODDS_RAW"].apply(extract_american_odds)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

# keep only valid rows
df = df[df["AMERICAN_ODDS"].notna() & df["LINE"].notna() & df["SIDE"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Loaded rows:", len(df))
print("Columns:", list(df.columns))
display(df.head(10)[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","IM_PROB_%","L5","L10","2025"]])

Loaded rows: 100
Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS_RAW', 'IM_PROB_%', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM/AW', 'AMERICAN_ODDS', 'SIDE', 'LINE', 'BOOK_PROB', 'DEC_ODDS']


/tmp/ipython-input-1748/1156004536.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw = raw.replace("", np.nan).dropna(how="all").fillna("")


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,IM_PROB_%,L5,L10,2025
1,Reed Sheppard,HOU at WAS,Rebounds + Assists,u8.5,-148,59.7%,80%,70%,83.1%
2,Cody Williams,DEN at UTA,Three Pointers Made,u0.5,-145,59.2%,100%,90%,71.7%
3,Reed Sheppard,HOU at WAS,Assists,u4.5,-163,62.0%,80%,80%,79.7%
4,Reed Sheppard,HOU at WAS,Rebounds + Assists,u7.5,111,47.4%,80%,70%,78%
5,Tari Eason,HOU at WAS,Three Pointers Made,u2.5,-185,64.9%,100%,90%,67.6%
6,Gui Santos,LAC at GSW,Rebounds,u5.5,-172,63.2%,60%,60%,81.6%
7,Julian Strawther,DEN at UTA,Rebounds,u3.5,-137,57.8%,80%,60%,78.4%
8,Nikola Jokić,DEN at UTA,Points + Assists,u40.5,-117,53.9%,80%,90%,63.6%
9,Tari Eason,HOU at WAS,Points,u14.5,-125,55.6%,100%,90%,70.3%
10,Tari Eason,HOU at WAS,Points + Assists,u16.5,-125,55.6%,100%,90%,70.3%


In [ ]:
import numpy as np
import pandas as pd

def pct_to_prob(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    try:
        v = float(s)
        return v/100.0 if v > 1 else v
    except:
        return np.nan

hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob)

# SAME weights as the prior run
weights = {
    "L5":0.20, "L10":0.25, "2025":0.20, "HM/AW":0.10, "H2H":0.05,
    "DVP_L5":0.08, "DVP_L10":0.07, "DVP_2025":0.04, "DVP_HM/AW":0.01
}

num = 0.0
den = 0.0
for c,w in weights.items():
    v = df[c].astype(float)
    num += w * v.fillna(np.nan)
    den += w * v.notna().astype(float)

df["HIT_PROB"] = (num / den).clip(0.01, 0.99)

print("HIT_PROB null:", df["HIT_PROB"].isna().sum(), "/", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","HIT_PROB"]].head(12))

HIT_PROB null: 0 / 100


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,HIT_PROB
1,Reed Sheppard,HOU at WAS,Rebounds + Assists,u8.5,-148,0.76312
2,Cody Williams,DEN at UTA,Three Pointers Made,u0.5,-145,0.71570
3,Reed Sheppard,HOU at WAS,Assists,u4.5,-163,0.74311
4,Reed Sheppard,HOU at WAS,Rebounds + Assists,u7.5,111,0.68889
5,Tari Eason,HOU at WAS,Three Pointers Made,u2.5,-185,0.80924
6,Gui Santos,LAC at GSW,Rebounds,u5.5,-172,0.68571
7,Julian Strawther,DEN at UTA,Rebounds,u3.5,-137,0.61795
8,Nikola Jokić,DEN at UTA,Points + Assists,u40.5,-117,0.82068
9,Tari Eason,HOU at WAS,Points,u14.5,-125,0.77508
10,Tari Eason,HOU at WAS,Points + Assists,u16.5,-125,0.77107


In [ ]:
import numpy as np

def pct_to_prob(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    try:
        v = float(s)
        return v/100.0 if v > 1 else v
    except:
        return np.nan

hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]

for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob)

weights = {
    "L5":0.20,
    "L10":0.25,
    "2025":0.20,
    "HM/AW":0.10,
    "H2H":0.05,
    "DVP_L5":0.08,
    "DVP_L10":0.07,
    "DVP_2025":0.04,
    "DVP_HM/AW":0.01
}

num = 0
den = 0
for c,w in weights.items():
    num += w * df[c]
    den += w * df[c].notna()

df["HIT_PROB"] = (num/den).clip(0.01,0.99)

display(df[["PLAYER","PROP","OUTCOME","HIT_PROB"]].head())

,PLAYER,PROP,OUTCOME,HIT_PROB
1,Reed Sheppard,Rebounds + Assists,u8.5,0.76312
2,Cody Williams,Three Pointers Made,u0.5,0.71570
3,Reed Sheppard,Assists,u4.5,0.74311
4,Reed Sheppard,Rebounds + Assists,u7.5,0.68889
5,Tari Eason,Three Pointers Made,u2.5,0.80924


In [ ]:
from scipy.stats import norm, poisson, nbinom

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p: return "three"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

def pick_dist(mkt, line):
    if mkt in ["three"]:
        return "poisson"
    if mkt in ["assists","rebounds"] and line <= 8.5:
        return "nbinom"
    if mkt in ["pra","pr","pa","ra"] or line >= 12:
        return "normal"
    return "normal"

df["DIST_USED"] = df.apply(lambda r: pick_dist(r.MKT, r.LINE), axis=1)

def prob_from_mu(line, mu, dist, side):
    if dist=="normal":
        sd = max(1.0, mu*0.35)
        return (1-norm.cdf(line, mu, sd)) if side=="over" else norm.cdf(line, mu, sd)
    if dist=="poisson":
        k=int(line)
        return (1-poisson.cdf(k, mu)) if side=="over" else poisson.cdf(k, mu)
    if dist=="nbinom":
        r=6.0
        p=r/(r+mu)
        k=int(line)
        return (1-nbinom.cdf(k, r, p)) if side=="over" else nbinom.cdf(k, r, p)

def invert_mu(target_p, line, dist, side):
    lo, hi = 0.05, max(2.0, line*3 + 10)
    for _ in range(50):
        mid = (lo+hi)/2
        pmid = prob_from_mu(line, mid, dist, side)
        if pmid < target_p:
            lo = mid if side=="over" else lo
            hi = hi if side=="over" else mid
        else:
            hi = mid if side=="over" else hi
            lo = lo if side=="over" else mid
    return (lo+hi)/2

df["PROJ_MU_BASE"] = df.apply(
    lambda r: invert_mu(r.HIT_PROB, r.LINE, r.DIST_USED, r.SIDE),
    axis=1
)

display(df[["PLAYER","PROP","OUTCOME","DIST_USED","HIT_PROB","PROJ_MU_BASE"]].head())

,PLAYER,PROP,OUTCOME,DIST_USED,HIT_PROB,PROJ_MU_BASE
1,Reed Sheppard,Rebounds + Assists,u8.5,normal,0.76312,6.796025
2,Cody Williams,Three Pointers Made,u0.5,poisson,0.71570,0.334494
3,Reed Sheppard,Assists,u4.5,nbinom,0.74311,3.284988
4,Reed Sheppard,Rebounds + Assists,u7.5,normal,0.68889,6.396876
5,Tari Eason,Three Pointers Made,u2.5,poisson,0.80924,1.498433


In [ ]:
# No ESPN layer for structure A (same as 3/1)
# Context shift derived from hit vs book delta

df["PROJ_MU_SHIFT"] = (df["HIT_PROB"] - df["BOOK_PROB"]) * df["PROJ_MU_BASE"]
df["PROJ_MU_CTX"]   = (df["PROJ_MU_BASE"] + df["PROJ_MU_SHIFT"]).clip(lower=0.05)

df["PROJ_PROB_CTX"] = df.apply(
    lambda r: prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE),
    axis=1
).clip(0.01,0.99)

df["BLEND_PROB"] = (0.6*df["PROJ_PROB_CTX"] + 0.4*df["HIT_PROB"]).clip(0.01,0.99)

In [ ]:
# Injury sensitivity proxy
def apply_mu(mu, direction):
    bump = mu*0.07
    return mu + bump if direction=="up" else max(0.05, mu - bump)

p_up=[]
p_dn=[]

for r in df.itertuples():
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"up"), r.DIST_USED, r.SIDE))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"down"), r.DIST_USED, r.SIDE))

df["INJ_SENS_ABS"] = np.abs(np.array(p_up)-np.array(p_dn))*100

df["SAFE_PROB"] = (
    df["BLEND_PROB"] -
    (df["INJ_SENS_ABS"]/100)*0.15
).clip(0.01,0.99)

df["EV_SAFE_1U"] = df["SAFE_PROB"]*df["DEC_ODDS"] - 1

def half_kelly(p, dec):
    b=dec-1
    k=(p*b-(1-p))/b
    return max(0,k/2)

df["UNITS"] = [half_kelly(p,d) for p,d in zip(df["SAFE_PROB"],df["DEC_ODDS"])]
df["UNITS"] = df["UNITS"].clip(0,0.5)

df_card = df[(df["EV_SAFE_1U"]>0) & (df["SAFE_PROB"]>df["BOOK_PROB"]+0.03)]
df_card = df_card.sort_values("EV_SAFE_1U",ascending=False)

display(df_card[["PLAYER","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","EV_SAFE_1U"]].head(10))

,PLAYER,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U
14,Bilal Coulibaly,Points + Rebounds + Assists,o15.5,-106,0.330376,0.835317,0.623351
58,Bilal Coulibaly,Points + Rebounds,o13.5,-101,0.298320,0.799323,0.590732
82,Keyonte George,Points + Assists,o23.5,115,0.230181,0.711356,0.529416
55,Keyonte George,Points + Rebounds + Assists,o25.5,100,0.237964,0.737964,0.475927
18,Bobby Portis,Points,o9.5,-125,0.266879,0.792781,0.427006
60,Bilal Coulibaly,Rebounds + Assists,o5.5,-118,0.240201,0.761652,0.407121
65,Bobby Portis,Points + Rebounds,o15.5,100,0.197271,0.697271,0.394542
33,Bilal Coulibaly,Points,o9.5,-113,0.222123,0.739082,0.393137
95,Keyonte George,Points + Rebounds,o20.5,-105,0.198242,0.705601,0.377603
72,Bilal Coulibaly,Points + Assists,o11.5,-120,0.212004,0.738185,0.353340


In [ ]:
top = df_card.head(5).copy().reset_index(drop=True)
top["RANK"] = range(1,len(top)+1)

print("NBA ELITE PROP CARD")
print("—")

for r in top.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

NBA ELITE PROP CARD
—
1) Bilal Coulibaly — HOU at WAS
Points + Rebounds + Assists o15.5 (-106) — 0.33u
2) Bilal Coulibaly — HOU at WAS
Points + Rebounds o13.5 (-101) — 0.30u
3) Keyonte George — DEN at UTA
Points + Assists o23.5 (115) — 0.23u
4) Keyonte George — DEN at UTA
Points + Rebounds + Assists o25.5 (100) — 0.24u
5) Bobby Portis — BOS at MIL
Points o9.5 (-125) — 0.27u


In [ ]:
import numpy as np

# --- Choose which George play to DROP ---
DROP_PROP_CONTAINS = "Points + Rebounds + Assists"   # drops George PRA, keeps George PA
DROP_PLAYER = "Keyonte George"

# Optional: enforce max 1 prop per player on the final card
ENFORCE_ONE_PER_PLAYER = True

df_rank = df_card.copy()

# 1) Start from a larger pool so replacements are available
pool = df_rank.sort_values("EV_SAFE_1U", ascending=False).copy()

# 2) Build an initial top 7 (gives us room to swap cleanly)
initial = pool.head(7).copy()

# 3) Drop the selected George play
mask_drop = (initial["PLAYER"].astype(str) == DROP_PLAYER) & (initial["PROP"].astype(str).str.contains(DROP_PROP_CONTAINS, case=False, regex=False))
initial = initial[~mask_drop].copy()

# 4) If there are still 2 George plays, keep only the best EV one
george_rows = initial[initial["PLAYER"].astype(str) == DROP_PLAYER].copy()
if len(george_rows) > 1:
    keep_idx = george_rows.sort_values("EV_SAFE_1U", ascending=False).index[0]
    initial = initial.drop(index=[i for i in george_rows.index if i != keep_idx])

# 5) If enforcing one per player, dedupe by player keeping best EV per player
if ENFORCE_ONE_PER_PLAYER:
    initial = (initial.sort_values("EV_SAFE_1U", ascending=False)
                     .drop_duplicates(subset=["PLAYER"], keep="first"))

# 6) Fill back up to 5 with next best non-included plays from full pool
selected = initial.sort_values("EV_SAFE_1U", ascending=False).copy()
need = 5 - len(selected)

if need > 0:
    already_players = set(selected["PLAYER"].astype(str))
    already_keys = set(zip(selected["PLAYER"].astype(str), selected["GAME"].astype(str), selected["PROP"].astype(str), selected["OUTCOME"].astype(str)))

    add_candidates = []
    for r in pool.itertuples():
        key = (str(r.PLAYER), str(r.GAME), str(r.PROP), str(r.OUTCOME))
        if key in already_keys:
            continue
        if ENFORCE_ONE_PER_PLAYER and str(r.PLAYER) in already_players:
            continue
        # don't add a second George play
        if str(r.PLAYER) == DROP_PLAYER:
            continue
        add_candidates.append(r)
        if len(add_candidates) >= need:
            break

    if len(add_candidates) < need:
        print("WARNING: Not enough replacement candidates to fill to 5 under current constraints.")
    else:
        add_df = pd.DataFrame([a._asdict() for a in add_candidates])
        selected = pd.concat([selected, add_df], ignore_index=True)

# 7) Final card + Discord print
selected = selected.sort_values("EV_SAFE_1U", ascending=False).head(5).copy().reset_index(drop=True)
selected["RANK"] = np.arange(1, len(selected)+1)

display(selected[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","EV_SAFE_1U"]])

print("NBA ELITE PROP CARD")
print("—")
for r in selected.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U
0,1,Bilal Coulibaly,HOU at WAS,Points + Rebounds + Assists,o15.5,-106,0.330376,0.835317,0.623351
1,2,Keyonte George,DEN at UTA,Points + Assists,o23.5,115,0.230181,0.711356,0.529416
2,3,Bobby Portis,BOS at MIL,Points,o9.5,-125,0.266879,0.792781,0.427006
3,4,Bub Carrington,HOU at WAS,Rebounds + Assists,o6.5,-144,0.217906,0.768776,0.302648
4,5,Brice Sensabaugh,DEN at UTA,Rebounds,o2.5,-173,0.229601,0.801905,0.265434


NBA ELITE PROP CARD
—
1) Bilal Coulibaly — HOU at WAS
Points + Rebounds + Assists o15.5 (-106) — 0.33u
2) Keyonte George — DEN at UTA
Points + Assists o23.5 (115) — 0.23u
3) Bobby Portis — BOS at MIL
Points o9.5 (-125) — 0.27u
4) Bub Carrington — HOU at WAS
Rebounds + Assists o6.5 (-144) — 0.22u
5) Brice Sensabaugh — DEN at UTA
Rebounds o2.5 (-173) — 0.23u


In [ ]:
import numpy as np

# Current locked players
locked_players = [
    "Bilal Coulibaly",
    "Keyonte George",
    "Bobby Portis",
    "Brice Sensabaugh"
]

# Remove Bub
df_adj = df_card[df_card["PLAYER"] != "Bub Carrington"].copy()

# Find next best play not already on card
replacement_pool = df_adj[
    ~df_adj["PLAYER"].isin(locked_players)
].sort_values("EV_SAFE_1U", ascending=False)

replacement = replacement_pool.head(1)

# Build final card
final_card = df_adj[
    df_adj["PLAYER"].isin(locked_players)
].copy()

final_card = pd.concat([final_card, replacement], ignore_index=True)
final_card = final_card.sort_values("EV_SAFE_1U", ascending=False).reset_index(drop=True)
final_card["RANK"] = np.arange(1, len(final_card)+1)

display(final_card[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

print("NBA ELITE PROP CARD")
print("—")
for r in final_card.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,1,Bilal Coulibaly,HOU at WAS,Points + Rebounds + Assists,o15.5,-106,0.330376
1,2,Bilal Coulibaly,HOU at WAS,Points + Rebounds,o13.5,-101,0.298320
2,3,Keyonte George,DEN at UTA,Points + Assists,o23.5,115,0.230181
3,4,Keyonte George,DEN at UTA,Points + Rebounds + Assists,o25.5,100,0.237964
4,5,Bobby Portis,BOS at MIL,Points,o9.5,-125,0.266879
5,6,Bilal Coulibaly,HOU at WAS,Rebounds + Assists,o5.5,-118,0.240201
6,7,Bobby Portis,BOS at MIL,Points + Rebounds,o15.5,100,0.197271
7,8,Bilal Coulibaly,HOU at WAS,Points,o9.5,-113,0.222123
8,9,Keyonte George,DEN at UTA,Points + Rebounds,o20.5,-105,0.198242
9,10,Bilal Coulibaly,HOU at WAS,Points + Assists,o11.5,-120,0.212004


NBA ELITE PROP CARD
—
1) Bilal Coulibaly — HOU at WAS
Points + Rebounds + Assists o15.5 (-106) — 0.33u
2) Bilal Coulibaly — HOU at WAS
Points + Rebounds o13.5 (-101) — 0.30u
3) Keyonte George — DEN at UTA
Points + Assists o23.5 (115) — 0.23u
4) Keyonte George — DEN at UTA
Points + Rebounds + Assists o25.5 (100) — 0.24u
5) Bobby Portis — BOS at MIL
Points o9.5 (-125) — 0.27u
6) Bilal Coulibaly — HOU at WAS
Rebounds + Assists o5.5 (-118) — 0.24u
7) Bobby Portis — BOS at MIL
Points + Rebounds o15.5 (100) — 0.20u
8) Bilal Coulibaly — HOU at WAS
Points o9.5 (-113) — 0.22u
9) Keyonte George — DEN at UTA
Points + Rebounds o20.5 (-105) — 0.20u
10) Bilal Coulibaly — HOU at WAS
Points + Assists o11.5 (-120) — 0.21u
11) Keyonte George — DEN at UTA
Assists o4.5 (-115) — 0.20u
12) Keyonte George — DEN at UTA
Points + Assists o22.5 (-115) — 0.20u
13) Keyonte George — DEN at UTA
Points o17.5 (-115) — 0.15u
14) Brice Sensabaugh — DEN at UTA
Rebounds o2.5 (-173) — 0.23u
15) Kyle Kuzma — BOS at MIL
Poin